#Benchmarking de Clustering em Transcriptômica Espacial: Visium HD em Câncer Colorretal

Este notebook utiliza dados públicos de referência ([Câncer Colorretal - Visium HD](https://www.10xgenomics.com/datasets/visium-hd-cytassist-gene-expression-libraries-of-human-crc)) para validar o pipeline analítico. Os dados são consolidados no formato SpatialData/Zarr, permitindo a integração de imagens histológicas e matrizes de expressão de alta resolução. O objetivo desta etapa é estabelecer um baseline de confiança estatística, comparando os resultados obtidos com os achados validados na literatura ([Oliveira et al., 2025](https://www.nature.com/articles/s41588-025-02193-3)).

# **1. Bibliotecas**
Foram utilizadas bibliotecas que possibilitam a eficiência no processamento de grandes volumes de dados e a precisão na identificação de padrões espaciais:

* **[spatialdata](https://spatialdata.scverse.org/en/latest/index.html)**: utilizada para armazenar e manipular dados multiômicos, integra imagens, coordenadas de pontos e tabelas de expressão gênica, como objetos AnnData. **[spatialdata_plot](https://github.com/scverse/spatialdata-plot)** para a geração de visualizações e gráficos espaciais e **[spatialdata_io](https://spatialdata.scverse.org/projects/io/en/latest/)** para leitura e gravação de vários formatos de dados ômicos espaciais. **[json](https://docs.python.org/3/library/json.html)** para criação do objeto `Spatialdata`.

* **[scanpy](https://scanpy.readthedocs.io/en/stable/index.html)**:  análise de dados de expressão gênica, pré-processamento, redução de dimensionalidade (PCA, UMAP), clustering e análise de expressão diferencial. Implementação em Python da biblioteca Harmony do R;

* **[geosketch](https://github.com/brianhie/geosketch)**: biblioteca desenvolvida por Hie et al., 2019, para downsampling geométrico de conjuntos de dados massivos de scRNA-seq preservando variabilidade biológica e estrutura geométrica da amostra, permitindo análises mais rápidas de agrupamento e visualização sem sobrecarregar memória RAM;

* **[gc](https://docs.python.org/3/library/gc.html)**: gerenciamento da memória RAM no ambiente, garantindo estabilidade durante o processamento de arquivos em alta resolução;

* **[PIL](https://github.com/python-pillow/Pillow)**: processamento de imagens;

* **[shapely](https://pypi.org/project/shapely/)**: manipulação e análise de objetos geométricos.

* **[geopandas](https://geopandas.org/en/stable/)**: trabalhar com dados vetoriais geoespaciais (pontos, linhas, polígonos).

* bibliotecas de Suporte: **[pandas](https://pandas.pydata.org/)**: e **[numpy](https://numpy.org/)** (np) para manipulação de matrizes;
* **[matplotlib](https://matplotlib.org/stable/tutorials/pyplot.html)** para a geração de visualizações e gráficos espaciais.


Para assegurar a reprodutibilidade deste estudo, as versões das bibliotecas foram fixadas. O processamento foi realizado no ambiente de nuvem Google Colab, utilizando instâncias com aceleradores de hardware (GPU) e RAM configurada para alta demanda.


In [ ]:
%%capture

!pip install uv
!uv pip install --system anndata==0.12.0 pydeseq2==0.5.2 squidpy==1.6.5 "spatialdata[extra]==0.4.0" geosketch==1.3 harmonypy==0.0.10 igraph==0.11.8 spatialdata-plot datashader dask-image "numpy<2"

In [ ]:
%%time

import numpy as np
import pandas as pd
import json
import gc #gerenciamento de memória RAM

import spatialdata as spd
import spatialdata_plot as splt
import spatialdata_io as so
import scanpy as sc
import scanpy.external as sce

import geopandas as gpd
from spatialdata.models import Image2DModel, TableModel, ShapesModel
from shapely.geometry import Polygon
from PIL import Image

import geosketch as sketch #downsampling de dados massivos
from pydeseq2.dds import DeseqDataSet #análise de expressão diferencial
from pydeseq2.ds import DeseqStats

import matplotlib.pyplot as plt
import seaborn as sns #necessário para o heatmap
from spatialdata.transformations import Identity, Scale

from google.colab import drive
import os
from IPython.display import Image as ShowImage, display
import requests
from scipy.stats import hypergeom

In [ ]:
#monta drive
drive.mount('/content/drive')

# **2. Ingestão de dados**

O [dataset de CRC](https://www.10xgenomics.com/datasets/visium-hd-cytassist-gene-expression-libraries-of-human-crc) utilizado neste trabalho foi disponibilizado publicamente pela 10x Genomics e compreende dados de quatro pacientes: P1 e P2 são amostras de tecido de câncer de cólon humano,P3 e P5 são amostra de tecido normal adjacente do cólon humano. Os dados foram processados originalmente pelo pipeline spaceranger count v4.0.1. A utilização de dados públicos da 10x Genomics garante a reprodutibilidade dos resultados.

In [ ]:
%%capture
!wget https://cf.10xgenomics.com/supp/spatial-exp/analysis-workshop/multisample_raw_data.tar.gz #download dos dados brutos
!tar xvzf multisample_raw_data.tar.gz #extração de arquivos
!rm multisample_raw_data.tar.gz #remoção do arquivo compactado para otimizar espaço

In [ ]:
#formata tamanhos dos arquivos para apresentação
def formatar_tamanho(tamanho_bytes):
    if tamanho_bytes == 0: return "0 B"
    unidades = ("B", "KB", "MB", "GB")
    i = 0
    while tamanho_bytes >= 1024 and i < len(unidades) - 1:
        tamanho_bytes /= 1024
        i += 1
    return f"{tamanho_bytes:.2f} {unidades[i]}"

#caminhos locais no colab
input_path = '/content/data/'
save_path = "/content/tabelas_processamento"

if not os.path.exists(save_path):
    os.makedirs(save_path)

#processa arquivos
files = [f for f in os.listdir(input_path) if not f.startswith('.')]
data_list = []

for file in files:
    file_path = os.path.join(input_path, file)
    if os.path.isfile(file_path):
        # identifica pacientes
        parts = file.split('_')
        paciente = parts[1] if len(parts) > 1 else "N/A"

        # identifica formato
        formato = os.path.splitext(file)[1]

        # de para de tipo de arquivo
        tipo = "Outros"
        if "cell_segmentations" in file:
            tipo = "Dados segmentados"
        elif "filtered_feature_cell_matrix" in file:
            tipo = "Matrizes agregadas (bins)"
        elif "scalefactors_json" in file:
            tipo = "Metadados posicionais"
        elif "tissue_hires_image" in file:
            tipo = "Imagem de microscopia"

        size_bytes = os.path.getsize(file_path)

        data_list.append({
            'Paciente': paciente,
            'Nome do arquivo': file,
            'Tipo de arquivo': tipo,
            'Formato': formato,
            'Tamanho': formatar_tamanho(size_bytes)
        })

#cria df ordenado por paciente e tipo de arquivo
df_final = pd.DataFrame(data_list)
df_final = df_final.sort_values(by=['Paciente', 'Tipo de arquivo']).reset_index(drop=True)

#salva tabela localmente
df_final.to_csv(f"{save_path}/inventario_arquivos.csv", index=False, sep=';', encoding='utf-8-sig')

display(df_final)

In [ ]:
#exporta para Drive
#drive.mount('/content/drive')
#save_path = "/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/tables"
#
##nomeia arquivo
#file_name = "Tabela 3. Descrição do conjunto de dados de referência.csv"
#full_path = os.path.join(save_path, file_name)
#
##export
#df_final.to_csv(full_path, index=False, sep=';', encoding='utf-8-sig')

In [ ]:
#visualização da amostra P1
#caminho_imagem = 'data/Cancer_P1_tissue_hires_image.png'
#display(ShowImage(filename=caminho_imagem, width=600))

#**3. Data Wrangling**

Para utilizar a biblioteca `spatialdata` é necessário converter os arquivos de saída do Visium HD em um arquivo Zarr. Esse formato possibilita a leitura eficiente de grandes volumes de dados, carregando somente o necessário para a RAM.


##**3.1. Funções auxiliares**





### **3.1.1. create_zarr**


Função que recebe os arquivos de saída brutos do Visium HD e os converte em um único arquivo Zarr para posterior análise espacial com a biblioteca SpatialData.
1. recebe caminhos de entrada;
2. carrega e prepara os dados;
3. define distemas de coordenadas e transformações;
4. processa a segmeentação celular;
5. integra a segmentação celular ao objeto AnnData;
6. cria objeto SpatialData;
7. grava tudo em um arquivo Zarr.

**inputs:** arquivos de saída brutos do processamento Visium HD:

* `image_path`: caminho para o arquivo de imagem;
* `count_matrix_path`: caminho para o arquivo da matriz de contagem de códigos de barras de recursos filtrados para segmentação das células;
* `scale_factors_path`: caminho para o arquivo JSON dos fatores de escala;
* `geojson_path`: caminho para o arquivo GEOJson de segmentação celular;
* `sample_name`: nome para o arquivo de saída do Zarr.

**output:** arquivo Zarr, que prepara os dados para análise espacial com bibliotecas como `spatialdata`

Para o becnhmarking, foi utilizado o arquivo `hires_image.png' para reduzir o tamanho do Zarr e economizar RAM, o que acelera o código. Porém, a imagem de microscópio em resolução total também pode ser usada, com ajustes no código para dimensionar corretamente o arquivo GeoJSON de segmentação celular. As imagens necessárias no arquivo Zarr final dependerão das análises subsequentes.


**Opção 1: Segmentação manual**

Função desenvolvida manualmente para desenhar o formato exato de cada célula.

Vantagem: biologicamente mais preciso, pois respeita a morfologia do tecido.

Desvantagem: computacionalmente mais complexo e pesado, exige mais RAM.


In [ ]:
def create_zarr(count_matrix_path,   #caminho para o arquivo da matriz de contagem
                image_path,          #caminho para o arquivo de imagem
                scale_factors_path,  #caminho para o arquivo JSON dos fatores de escala
                geojson_path,        #caminho para o arquivo GeoJSON de segmentação das células
                sample_name          #nome para o arquivo de saída do Zarr
):
    # 1. recebe caminhos de entrada
    print(sample_name)

    # 2. carrega e prepara os dados
    COUNT_MATRIX_PATH = count_matrix_path
    IMAGE_PATH = image_path
    SCALE_FACTORS_PATH = scale_factors_path
    GEOJSON_PATH = geojson_path

    # Load AnnData
    adata = sc.read_10x_h5(COUNT_MATRIX_PATH)
    adata.var_names_make_unique()
    adata.obs['sample'] = sample_name

    # AJUSTE para o mapeamento do GeoJSON
    adata.obs.index = sample_name + "_" + adata.obs.index.astype(str)

    # Load and preprocess image data
    image_data = np.array(Image.open(IMAGE_PATH))
    if image_data.ndim == 2:
        image_data = image_data[np.newaxis, :, :] # Add channel dimension for grayscale
    elif image_data.ndim == 3:
        image_data = np.transpose(image_data, (2, 0, 1)) # (H, W, C) -> (C, H, W) for spatialdata

    # Load scale factors
    with open(SCALE_FACTORS_PATH, 'r') as f:
        scale_data = json.load(f)

    # Load GeoJSON data
    with open(GEOJSON_PATH, 'r') as f:
        geojson_data = json.load(f)

    # 3. define sistemas de coordenadas e transformações
    hires_scale = scale_data['tissue_hires_scalef']
    shapes_transformations = {
       "downscale_to_hires": Scale(np.array([hires_scale, hires_scale]), axes=("x", "y"))
    }
    image_transformations = {
        "downscale_to_hires": Identity()
    }

    # 4. processa a segmentação celular
    geojson_features_map = {
        f"{sample_name}_cellid_{feature['properties']['cell_id']:09d}-1": feature
        for feature in geojson_data['features']
    }

    geometries = []
    cell_ids_ordered = []

    for obs_index_str in adata.obs.index:
        feature = geojson_features_map.get(obs_index_str)
        if feature:
            polygon_coords = np.array(feature['geometry']['coordinates'][0])
            geometries.append(Polygon(polygon_coords))
            cell_ids_ordered.append(obs_index_str)
        else:
            geometries.append(None)
            cell_ids_ordered.append(obs_index_str)

    valid_indices = [i for i, geom in enumerate(geometries) if geom is not None]
    geometries = [geometries[i] for i in valid_indices]
    cell_ids_ordered = [cell_ids_ordered[i] for i in valid_indices]

    # 5. integra a segmentação celular ao objeto AnnData
    shapes_gdf = gpd.GeoDataFrame({
        'cell_id': cell_ids_ordered,
        'geometry': geometries
    }, index=cell_ids_ordered)

    # AJUSTE para a compatibilidade de dtypes entre Pandas e SpatialData
    adata.obs['cell_id'] = adata.obs.index.astype(str)

    adata.obs['region'] = sample_name + '_cell_boundaries'
    adata.obs['region'] = adata.obs['region'].astype('category')
    adata = adata[shapes_gdf.index].copy()

    IMAGE_KEY =  sample_name + '_hires_tissue_image'
    TABLE_KEY =  'segmentation_counts'
    SHAPES_KEY = sample_name + '_cell_boundaries'

    # 6. cria objeto SpatialData
    sdata = spd.SpatialData(
        images={
            IMAGE_KEY: Image2DModel.parse(image_data, transformations=image_transformations)
        },
        tables={
            TABLE_KEY: TableModel.parse(
                adata,
                region=SHAPES_KEY,
                region_key='region',
                instance_key='cell_id'
            )
        },
        shapes={
            SHAPES_KEY: ShapesModel.parse(shapes_gdf, transformations=shapes_transformations)
        }
    )

    # 7. grava tudo em um arquivo Zarr.
    sdata.write(sample_name, overwrite=True)
    del sdata
    gc.collect()

**Opção 2: Square-Bins**

Divide o tecido em bins. Ideal para análises exploratórioas rápidas em grandes datasets. Mais rápida e simplificada pois usa uma função pronta da biblioteca (`so.visium_hd`)




In [ ]:
def create_zarr_bins(
                path_to_outputs,
                zarr_name,
                bin_size
):
    print(zarr_name)
    sdata = so.visium_hd(path=path_to_outputs,
                         load_all_images=True, bin_size=bin_size)
    sdata.write(zarr_name, overwrite=True)
    del sdata

**Opção 3: Segmentação Automática**

Utiliza os recursos nativos e modernos da biblioteca para extrair automaticamente a segmentação de núcleos. Combina a precisão da segmentação com a facilidade de automação das ferramentas mais recentes.

A utilização da so.visium_hd resulta em nomes diferentes no objeto SpatialData para imagens, formas, tabelas e sistemas de coordenadas, o que requer ajuste no código.

In [ ]:
#def create_zarr(
#                path_to_outputs,
#                zarr_name,
#                bin_size
#):
#    print(zarr_name)
#    sdata = so.visium_hd(path=path_to_outputs,
#                         load_all_images=True,
#   load_segmentations_only=True,
#   			        load_nucleus_segmentations=True,
#)
#    sdata.write(zarr_name, overwrite=True)
#    del sdata

### **3.1.2. crop0**

Garante que as imagens geradas pela análise sejam recortadas para a região de interesse, alinhando-se à área de captura Visium HD de cada amostra.

inputs:
* `SpatialData` (`x`)
* um sistema de coordenadas alvo (`crs`)
* um dicionário de caixa delimitadora (`bbox`), que é uam forma de representar uma região retangular em um espaço 2D com um dicionário Python. Contém coordenadas mínimas e máximas para os eixos x e y, que definem o limite da caixa. A função pressupõe que o dicionário `bbox` foi criado usando `spd.get_extent` com o mesmo sistema de coordenadas.

Internamente, a função chama `spd.bounding_box_query` e usa as coordenadas mínimas e máximas do dicionário para selecionar os dados do objeto `SpatialData` que se encontram dentro desse retângulo definido. Isso garante que as visualizações ou análises subsequentes se concentrem apenas na parte relevante dos dados. Isso é necessário porque a imagem do microscópio costuma ser maior do que a área de captura de expressão gênica do Visium HD.

In [ ]:
def crop0(x,crs,bbox):
    return spd.bounding_box_query(
        x,
        min_coordinate=[bbox['x'][0], bbox['y'][0]],
        max_coordinate=[bbox['x'][1], bbox['y'][1]],
        axes=("x", "y"),
        target_coordinate_system=crs,
    )

##3.2. Processamento em lote (aplicação create_zarr)

Aplicação da função `create_zarr_manual_segmentation` aos datasets dos pacientes P1, P2, P2 e P5.

É criado um dicionário em que cada nome de amostra é uma chave e o valor é uma lista contendo:
* matriz de contagem (.h5);
* imagem de alta resolução (.png)
* fatores de de escala, para que os resultados da segmentação celular possam ser sobrepostos à imagem do tecido (.json)
* arquivo de segmentação celular, para que os resultados da segmentação celular possam ser visualizados na imagem do tecido (.geojson)
* nome desejado para o arquivo Zarr.

In [ ]:
%%time
# Create and save Zarr files for the cell segmentation outputs.
#cria dicionário de amostras
samples = {
    "Colon_Cancer_P1":["data/Cancer_P1_filtered_feature_cell_matrix.h5",
                      "data/Cancer_P1_tissue_hires_image.png",
                      "data/Cancer_P1_scalefactors_json.json",
                      "data/Cancer_P1_cell_segmentations.geojson",
                      "Colon_Cancer_P1"],
    "Colon_Cancer_P2":["data/Cancer_P2_filtered_feature_cell_matrix.h5",
                      "data/Cancer_P2_tissue_hires_image.png",
                      "data/Cancer_P2_scalefactors_json.json",
                      "data/Cancer_P2_cell_segmentations.geojson",
                      "Colon_Cancer_P2"],
    "Colon_Normal_P3":["data/Norm_P3_filtered_feature_cell_matrix.h5",
                      "data/Norm_P3_tissue_hires_image.png",
                      "data/Norm_P3_scalefactors_json.json",
                      "data/Norm_P3_cell_segmentations.geojson",
                      "Colon_Normal_P3"],
    "Colon_Normal_P5":["data/Norm_P5_filtered_feature_cell_matrix.h5",
                      "data/Norm_P5_tissue_hires_image.png",
                      "data/Norm_P5_scalefactors_json.json",
                      "data/Norm_P5_cell_segmentations.geojson",
                      "Colon_Normal_P5"],
}
#loop percorre o dicionário e chama create_zarr
print("Saving zarr files")
for key, inputs in samples.items():
    create_zarr(count_matrix_path=inputs[0],
                image_path=inputs[1],
                scale_factors_path=inputs[2],
                geojson_path=inputs[3],
                sample_name=inputs[4])

del samples, inputs, key
gc.collect() #limpar a RAM entre pacientes

Foram criadas 4 pastas independentes, uma para cada paciente. No próximo passo, elas serão unidas em um único objeto spatialdata.

##3.3. Criação objeto SpatialData

Um objeto `Spatialdata` é composto por:

* **Images**: camada raster contendo imagens do CytAssist e de microscopia que fornecem contexto espacial, podem ser acessadas por `sdata.images`. Armazenadas em coordenadas tridimensionais. c=3 indica sistema RGB, y,x a resolução em pixels.;

* **Shapes**: camada vetorial com polígonos resultantes da segmentação celular (geojson), que representam regiões de interesse, células ou pontos. Armazena a geometria dos bins de segmentação (polígonos). 2D é o plano bidimensional da fatia de tecido. No Visium HD, geralmente regiões agrupadas ou segmentações celulares. Podem ser acessadas por `sdata.shapes`;

* **Tables**:  matriz de expressão gênica unificada (AnnData), é a camada estatística. Consolida os dados do experimento em um só sistema de coordenadas (downscale_to_hires). Vincula as contagens moleculares aos polígonos da camada Shapes e às coordenadas da camada Images. Se o código for pintar um gene específico, ele sabe em qual pixel da imagem e em qual polígono do shape ele deve aplicar a cor. objeto `AnnData` associado aos elementos espaciais, geralmente com dados de expressão gênica, metadados das células e resultados computacionais. Objeto usado para análises subsequentes, que pode ser acessado por `sdata.tables` e contém:
  * .X: matriz de dados primária;
  * .obs: metadados das observações, como por exemplo ID da amostra;
  * .var: metadados de variáveis, como por exemplo nomes dos genes;
  * .obsm: anotações multidimensionais, como por exemplo PCA, UMAP;
  * .layers: representações alternativas de X.

Após o processamento de ETL, os arquivos Zarr são consolidados em um objeto `Spatialdata` através da função `read_zarr`.
A coluna `sample` é adicionada a cada tabela `AnnData` dentro do objeto `Spatialdata` para identificar a qual amostra pertence.

A função `var_names_make_unique` garante que os nomes dos genes sejam únicos.

In [ ]:
%%time

# caminhos dos arquivos Zarr criados anteriormente
visium_hd_zarr_paths = {
    "Cancer_P1": "./Colon_Cancer_P1",
    "Cancer_P2": "./Colon_Cancer_P2",
    "Normal_P3": "./Colon_Normal_P3",
    "Normal_P5": "./Colon_Normal_P5"
}

# carrega amostras
sdatas = []
for key, path in visium_hd_zarr_paths.items():
    sdata = spd.read_zarr(path)

    for table in sdata.tables.values():
        table.var_names_make_unique()   # garante nomes de genes únicos
        table.obs["sample"] = key       # identificação da amostra

    sdatas.append(sdata)
    del sdata, table
    gc.collect()

# concatena amostras
concatenated_sdata = spd.concatenate(sdatas, concatenate_tables=True)

# carrega as imagens na RAM e revalida o esquema para evitar o SchemaError
for img_name in list(concatenated_sdata.images.keys()):
    # .compute() traz o dado para a RAM; Image2DModel.parse() re-empacota no formato correto
    img_data = concatenated_sdata.images[img_name].compute()
    concatenated_sdata.images[img_name] = Image2DModel.parse(img_data)

# checkpoint sobrescrevendo o anterior no Google Drive
path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'
os.makedirs(path_crc, exist_ok=True)
concatenated_sdata.write(os.path.join(path_crc, 'sdata_100k_raw.zarr'), overwrite=True)

# limpa variáveis temporárias mas mantém o objeto principal na memória
del sdatas, visium_hd_zarr_paths, key, path
gc.collect()

print("---------------------------------")
print(concatenated_sdata)

**Dos resultados obtidos:**

Objeto consolidado contendo 4 images, 4 shapes, um de cada amostra. 1 table segmentation_counts.

nº de observações = nº total de linhas, exatamente igual à soma de todos os shapes (P1+P2+P3+P5), o que confirma que o SpatialData empilhou corretamente em uma única tabela.

colunas = quantidade de genes diferentes que o sequenciamento detectou (18085). O objeto unifica os genes. Se um gene foi detectato em só um paciente, todos passam a ter uma coluna pra esse gene mas ela fica com zero quando ele não é encontrado.

Visualização do resultado

In [ ]:
#tabela de contagens
table = concatenated_sdata.tables['segmentation_counts']

#total de UMI por célula/bin
try:
    table.obs['total_counts'] = np.ravel(table.X.sum(axis=1))
except:
    table.obs['total_counts'] = table.X.sum(axis=1).compute()

#agrupa por amostra
tabela_validacao = table.obs.groupby('sample').agg(
    bins=('total_counts', 'count'),
    total_UMI=('total_counts', 'sum')
).reset_index()

#coluna de pacientes
tabela_validacao['Amostra'] = tabela_validacao['sample'].str[-2:]

#conta genes por amostra
def contar_genes_meus(sample_id):
    sub = table[table.obs['sample'] == sample_id].X
    return int(np.sum(np.ravel(sub.sum(axis=0)) > 0))

tabela_validacao['genes'] = tabela_validacao['sample'].apply(contar_genes_meus)
tabela_validacao['fonte'] = 'Validação'

#ordena
tabela_validacao = tabela_validacao[['Amostra', 'bins', 'total_UMI', 'genes', 'fonte']]

print("Dados da validação")
display(tabela_validacao.style.hide(axis='index').format(precision=0, thousands="."))

Validação com a literatura

In [ ]:
#literatura

#carrega arquivo Source Data Extended Data Fig. 10 (download CSV) - Raw UMIs per gene, statistical source data.
url_nature = "https://static-content.springer.com/esm/art%3A10.1038%2Fs41588-025-02193-3/MediaObjects/41588_2025_2193_MOESM17_ESM.csv"
df_nature_raw = pd.read_csv(url_nature)

df_nature_hd = df_nature_raw[df_nature_raw['sample'].str.contains('Visium_HD', case=False)].copy() #Filtra Visium_HD
df_nature_hd['Amostra'] = df_nature_hd['sample'].str[-2:] #cria coluna paciente

#tabela agrupada
tabela_literatura = df_nature_hd.groupby('Amostra').agg(
    total_UMI=('num_umis', 'sum')
).reset_index()

#calcula UMI
genes_nature = df_nature_hd[df_nature_hd['num_umis'] > 0].groupby('Amostra')['feature_name'].nunique()
tabela_literatura['genes'] = tabela_literatura['Amostra'].map(genes_nature)

#cria colunas para comparação
tabela_literatura['bins'] = "-" #o artigo não tem bins brutos
tabela_literatura['fonte'] = 'Literatura'

#ordena colunas
tabela_literatura = tabela_literatura[['Amostra', 'bins', 'total_UMI', 'genes', 'fonte']]

print("Dados da literatura")
display(tabela_literatura.style.hide(axis='index').format(precision=0, thousands="."))

In [ ]:
#concatena tabelas para comparação
display(pd.concat([tabela_literatura, tabela_validacao], ignore_index=True)
        .sort_values(by=['Amostra', 'fonte'])
        .reset_index(drop=True))

In [ ]:
#export para o drive
#drive.mount('/content/drive', force_remount=True)
#caminho_tabelas = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/tables'
#
#if os.path.exists(caminho_tabelas):
#    tabela_final = (pd.concat([tabela_literatura, tabela_validacao], ignore_index=True)
#                    .sort_values(by=['Amostra', 'fonte'])
#                    .reset_index(drop=True))
##export
#tabela_final.to_csv(f"{caminho_tabelas}/3.3.count_genes.csv", sep=';', index=False, encoding='utf-8-sig')

Mesmo após a conversão de matrizes h5, imagens e geojsons para Zarr/Spatial data, verifica-se a integridade das informações pela quantidade de genes na mesma ordem de grandeza entre o que demonstra a literatura e o que foi obtido ao rodar o pipeline. Isso valida a eficiência da conversão para o formato Spatialdata e assegura que a informação foi preservada para as análises espaciais.

A diferença entre o total_UMI para o mesmo paciente se dá por conta da função create_zarr, que mantém na tabela apenas os dados que se encontram dentro dos polígonos (adata = adata[shapes_gdf.index].copy()), enquanto que os dados da literatura usam o total da lâmina.

- bins - unidade de medida do espaço, quadradinhos com 2µm ou 8µm de lado;
- total_UMI - Unique Molecular Identifier, contagem total de moléculas de RNA detectadas;
- genes - espécies diferentes de genes detectados, prova a integridade biológica. como o número se aproxima da literatura, o pipeline enxerga a mesma diversidade.

A volumetria dos dados foi avaliada através de três métricas fundamentais: a densidade amostral (bins), o volume total de moléculas capturadas (Total_UMI) e a riqueza do transcriptoma detectado (genes). A paridade observada na detecção de genes entre a validação e a literatura confirma que o pipeline preserva a diversidade biológica das amostras.


## **3.4. Controle da qualidade (QC)**

Etapa de filtragem dos dados para evitar ruídos técnicos na análise.

É feita remoção de bins de baixa qualidade através da visualização da distribuição total de UMI (Unique Molecular Identifiers, moléculas de RNA). Estima-se o corte adequado de bins vazios ou pouco populados.

É mensurada a quantidade de células mitocondriais e, de acordo com a literatura, é adotado o limiar de 20%. Células grandes sempre terão mais mitocôndrias que células pequenas, então justifica-se a utilização do percentual. O que indica de fato morte celular é a proporção desses genes.

Clustering é bastante sensível a outliers, o que justifica a importância dessa seção.

O uso da escala logarítmica nos gráficos se justifica pela natureza desigual da distribuição de RNA.

In [ ]:
#tabela de contagem (AnnData) dentro do objeto consolidado (Spatialdata)
adata = concatenated_sdata.tables['segmentation_counts'] # we link the AnnData Table in the SpatialData object to the variable adata to make the code easier to read
display(adata)

In [ ]:
#identificação dos genes mitocondriais, que começam com "MT-"
adata.var["mt"] = adata.var_names.str.startswith(("MT-", "mt-"))
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True, percent_top=None)

#visualização do total de UMIs por amostra
plt.figure(figsize=(10, 5)) #proporção da imagem
ax = sc.pl.violin(adata, keys=["log1p_total_counts"], groupby="sample", stripplot=False, inner="box", show=False)
ax.set_xlabel("Amostra") #rótulo do eixo x
ax.set_ylabel("Contagem de UMI (Log1p)") #rótulo do eixo y
plt.axhline(y=4, color='r', linestyle='--', label="limite inferior") #linhas em y=4 y=8 representam limites sugeridos de corte
plt.axhline(y=8, color='r', linestyle='--', label="limite superior")
plt.title("Distribuição de UMI (Log1p) por Amostra") #deixa claro que a escala do eixo x não é linear e sim logarítmica
plt.show()

#visualização da quantidade de genes por amostra
plt.figure(figsize=(10, 5)) #proporção da imagem
ax = sc.pl.violin(adata, keys=["log1p_n_genes_by_counts"], groupby="sample", stripplot=False, inner="box", show=False)
ax.set_xlabel("Amostra") #rótulo do eixo x
ax.set_ylabel("Número de genes únicos (Log1p)") #rótulo do eixo y
plt.title("Distribuição de genes únicos por amostra")
plt.show()

#visualização da distribuição de genes mitocondriais em números absolutos
plt.figure(figsize=(10, 5)) #proporção da imagem
ax = sc.pl.violin(adata, keys=["log1p_n_genes_by_counts"], groupby="sample", stripplot=False, inner="box", show=False)
ax.set_xlabel("Amostra") #rótulo do eixo x
ax.set_ylabel("Número de genes mitocondriais") #rótulo do eixo y
sc.pl.violin(adata=adata, keys=["log1p_total_counts_mt"], groupby="sample", stripplot=False, inner="box",show=False)
plt.title("Número de genes mitocondriais por amostra")
plt.show()

#visualização do percentual de genes mitocondriais em relação ao total da célula, indicador de morte celular
plt.figure(figsize=(10, 5)) #proporção da imagem
ax = sc.pl.violin(adata, keys=["pct_counts_mt"], groupby="sample", stripplot=False, inner="box", show=False)
ax.set_xlabel("Amostra") #rótulo do eixo x
ax.set_ylabel("Genes Mitocondriais (%)") #rótulo do eixo y
plt.title("Percentual de genes mitocondriais por amostra")
plt.show()

plt.close('all')

Bins com menos de 53 moléculas (log1p=4) ou mais de 2.979 moléculas (log1p=8) serão descartados. Isso porque bins com pouquíssimo RNA (53) geralmente são áreas em que o sensor não detectou o tecido corretamente, ou são ruído de fundo. Áreas com RNA demais (2.979) pode ser um acúmulo de reagentes ou várias células sobrepostas, dado que no Visium HD um bin é minúsculo, com 2 $\mu$m. Esses outliers afetariam o clustering e serão removidos em seguida com `filter_cells`do `Scanpy`.

Os limites de contagem e um número de bins em que um gene deve estar presente varia entre experimentos e devem ser determinados empiricamente.

Esses limiares foram escolhidos "visualmente" a partir da distribuição dos dados.


In [ ]:
#estimativa do ponto de corte - converte os limites log para contagens reais
import numpy as np
min_counts = np.expm1(4).astype("int")
max_counts = np.expm1(8).astype("int")

#filtragem de genes e células
sc.pp.filter_genes(adata, min_cells=50)
sc.pp.filter_cells(adata, min_counts=min_counts) #limite mínimo de UMIs
sc.pp.filter_cells(adata, max_counts=max_counts) #limite máximo de UMIs

#backup dos dados filtrados para uso posterior. .copy() garante que os valores brutos de contagem sejam preservados.
adata.layers["filtered_counts"] = adata.X.copy()

#salva checkpoint no drive
path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'
adata.write_h5ad(os.path.join(path_crc, 'adata_100k_qc_filtered.h5ad'))

#amostra temporária apenas para visualização
idx_viz = np.random.choice(adata.n_obs, size=min(50000, adata.n_obs), replace=False)
adata_viz = adata[idx_viz].copy()

#visualização para confirmar o corte

#visualização do total de UMIs por amostra
plt.figure(figsize=(10, 5)) #proporção da imagem
ax = sc.pl.violin(adata_viz, keys=["log1p_total_counts"], groupby="sample", stripplot=False, inner="box", show=False)
ax.set_xlabel("Amostra") #rótulo do eixo x
ax.set_ylabel("Contagem de UMI (Log1p)") #rótulo do eixo y
plt.axhline(y=4, color='r', linestyle='--')
plt.axhline(y=8, color='r', linestyle='--')
plt.title("Distribuição de UMI após filtragem")
plt.show()

#visualização da quantidade de genes por amostra
plt.figure(figsize=(10, 5)) #proporção da imagem
ax = sc.pl.violin(adata_viz, keys=["log1p_n_genes_by_counts"], groupby="sample", stripplot=False, inner="box", show=False)
ax.set_xlabel("Amostra") #rótulo do eixo x
ax.set_ylabel("Número de genes únicos (Log1p)") #rótulo do eixo y
plt.title("Distribuição de genes únicos após filtragem")
plt.show()

#visualização do percentual de genes mitocondriais em relação ao total da célula, indicador de morte celular
plt.figure(figsize=(10, 5)) #proporção da imagem
ax = sc.pl.violin(adata_viz, keys=["pct_counts_mt"], groupby="sample", stripplot=False, inner="box", show=False)
ax.set_xlabel("Amostra") #rótulo do eixo x
ax.set_ylabel("Genes Mitocondriais (%)") #rótulo do eixo y
plt.title("Percentual de genes mitocondriais após filtragem")
plt.show()

#limpeza final da memória
del adata_viz, idx_viz, max_counts, min_counts
gc.collect()

In [ ]:
#visualização para não repetir violinos

#busca contagem original no drive
path_raw = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc/sdata_100k_raw.zarr'
sdata_temp = spd.read_zarr(path_raw)
contagem_pre = sdata_temp.tables['segmentation_counts'].obs['sample'].value_counts()

#busca contagem depois do corte
contagem_pos = adata.obs['sample'].value_counts()

#df comparativo
df_qc = pd.DataFrame({
    'Amostra': contagem_pre.index,
    'Bins originais': contagem_pre.values,
    'Bins filtrados': [contagem_pos.get(s, 0) for s in contagem_pre.index]
})

#calcula impacto
df_qc['Retenção (%)'] = (df_qc['Bins filtrados'] / df_qc['Bins originais'] * 100).round(2)
df_qc['Perda (%)'] = (100 - df_qc['Retenção (%)']).round(2)

print("\nQC: impacto da filtragem")
display(df_qc.style.hide(axis='index').background_gradient(subset=['Perda (%)'], cmap='Reds'))

total_removido = df_qc['Bins originais'].sum() - df_qc['Bins filtrados'].sum()
print(f"\nTotal removido: {total_removido:,} bins.".replace(',', '.'))

#limpeza
del sdata_temp
gc.collect()

A retenção ficou entre 92 e 95%, o que demonstra que a qualidade dos dados é alta. Quase todo o RNA captado pelo sensor pertence ao tecido e não a ruídos de fundo. Se a perda fosse muito grande em uma amostra, poderia ser um alerta de que aquela amostra apresentou problemas técnicos no laboratório.

Caso o algortitmo performe mal, o percentual de perda da amostra no QC pode ajudar a explicar o motivo. Um baixo percentual aqui poderia indicar menor qualidade no RNA inicial.
Caso o Clustering não consiga separar as células adequadamente, se a retenção aqui for alta, pode-se dizer que o problema não é o dado e sim a sensibilidade do algoritmo para detectar padrões espaciais nesta resolução.

No Visium HD, a perda costuma ficar entre 10 e 25%. Se for menor que 5%, talvez o filtro esteja leve demais. Se for maior que 50%, muito agressivo. Como o Visium HD é muito denso, é esperado que bins vazios ou com pouca captura sejam deletados. Uma perda de 5% a 8% é o sinal de uma amostra de qualidade. Na documentação técnica da 10x Genomics, nos guias de análise do Space Ranger e dos datasets de demonstração do Visium HD, a fabricante informa que espera-se que a maioria dos bins (75-80%) estejam sobre o tecido.

A alta taxa de retenção de bins (superior a 92% em todas as amostras) após a filtragem de QC assegura alta qualidade do conjunto de dados.

# **4. Experimentos**

Foram realizados 6 experimentos, sendo eles:
* 4.1. amostra de 100k bins;
* 4.2. amostra de 100k bins + HVG;
* 4.3. amostra de 100k bins + HVG + normalização com Harmony;
* 4.4. amostra de 200k bins + HVG + normalização com Harmony;
* 4.5. base full.

Foi feito o processo de normalização das contagens filtradas e PCA para redução da dimensionalidade através de funções de pré-processamento do `Scanpy`:

* `normalize_total`: a normalização dos dados faz com que cada bin seja ajustado para a medida do dataset, assim, compara-se a proporção dos genes e não o volume bruto de captura. `target_sum=None` para que, após a normalização, cada bin de segmentação celular tenha um total de contagem de UMIs igual à mediana da contagem total de UMIs para todos os bins de segmentação celular antes da normalização.;
* `log1p`: transforma os dados para escala logaritmica;
* `pca`: resume os genes em componentes principais, que explicam a maior parte da variação dos dados. Por padrão, `pca` do `Scanpy` gera os 50 primeiros componentes principais, mas isso pode ser setado. As etapas de normalização e redução de dimensionalidade são realizadas no objeto `SpatialData` sem copiar o objeto `AnnData`. Os resultados são armazenados diretamente no objeto `sdata_concatenate`, uma vez que a variável `adata` está vinculada a ele.

Existem duas possibilidades:
* utilizar `highly_variable_genes` do scanpy para identificar genes altamente variáveis, para que só eles sejam usados no PCA;

* utilizar o dataset completo, escolha que faz sentido porque no Visium HD a qualidade dos dados é muito alta. Utilizar o transcriptoma completo preserva a resolução máxima da tecnologia.

Elbow plot: O objetivo é selecionar o número de PCs que capturam a maior parte da variância. Após o cotovelo, adicionar mais componentes traz pouca informação nova e pode confundir os algoritmos de clustering e UMAP.



## **4.1. Amostra de 100k**

### **4.1.2. PCA**



In [ ]:
%%time

# restauração
adata.X = adata.layers["filtered_counts"].copy()

# limpeza de metadados
if 'log1p' in adata.uns:
    del adata.uns['log1p']

# normalização e transformação
sc.pp.normalize_total(adata, target_sum=None) # ajusta pela mediana
sc.pp.log1p(adata) # estabiliza a variância

# PCA no adata completo 650k
sc.tl.pca(adata, svd_solver='arpack')

#checkpoint
path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'
os.makedirs(path_crc, exist_ok=True)
adata.write(os.path.join(path_crc, 'adata_100k_pca.h5ad'))

# gráfico de cotovelo
plt.figure(figsize=(8, 4))
ax = sc.pl.pca_variance_ratio(adata, log=True, n_pcs=50, show=False)
plt.title("Gráfico de cotovelo")
plt.show()

gc.collect()

preprocessed_adata.h5ad: salva o objeto AnnData, contendo:
* a matriz de contagem (X);
* os metadados das células (obs);
* os metadados dos genes (var)
* as camadas de processamento (layers);
* os resultados das análises (como as coordenadas do PCA e do UMAP).

Importância: Como o processamento de 650k bins é pesado, esse arquivo serve como um checkpoint.

### 4.1.2. Geosketch

Amostragem para ganahr tempo e economizar RAM. Diferente de uma amostragem aleatória simples, esta técnica utiliza a geometria do espaço latente gerada pelo PCA para selecionar amostras de todo o contexto transcriptômico. Assim, bins situados em áreas de baixa densidade (populações raras) passam a ter representatividade proporcionalmente maior do que teriam em uma amostragem uniforme convencional

In [ ]:
#utilização do geosketch para reduzir carga da RAM
N_SAMPLES = 100000 #redução de 650k células para 100k
sketch_index = sketch.gs(adata.obsm['X_pca'], N_SAMPLES, replace=False)
adata = adata[sketch_index].copy() #cria novo objeto e libera o anterior da memória

#checkpoint
adata.write(os.path.join(path_crc, 'adata_100k_sketched.h5ad'))
gc.collect() #limpa RAM

7 minutos para a amostragem de 100k

### 4.1.3. Clustering e UMAP

Após a padronização dos dados e PCA, será feita a clusterização e visualização dos dados.
A função `neighbors` do `Scanpy` gera a matriz de distância entre um vizinho e um grafo de vizinhança, que é usado pela função `leiden` para clusterizar os dados. Então, a função `umap` é usada para visualizar os resultados.

-------------

**Clustering:** o método Leiden destaca-se por realizar ajustes repetidos (processo iterativo) para encontrar a melhor divisão dos grupos, garantindo que as comunidades identificadas sejam bem conectadas entre si e biologicamente coerentes. (Traag et al., 2019).

Trata-se de um algoritmo de detecção de comunidades. Primeiro, o pipeline cria um gráfo de vizinhos (KNN), em que cada bin é um nó conectado aos seus vizinhos mais similares no espaço latente. Então, o Leiden identifica grupos de nós que estão mais conectados entre si do que com o resto do grafo. Utiliza um processo de refinamento iterativo das comunidades.

Parâmetros:
* n_neighbors: define quantas células próximas o algoritmo deve olhar para definir onde colocar um ponto. Valores mais baixos (<=15) fazem com que o algortimo agrupe apenas células muito parecidas, quebrando em ilhas menores e mais específicas. Valores altos (>50) fazem com que o algortimo tente agrupar tudo, misturando grupos e formando uma nuvem, ignorando o ruído de células isoladas;

* min_dist: controla quão próximos os pontos podem ficar uns dos outros. Valores baixos (0.1 a 0.3) permitem que os pontos se aglomerem com muita densidade, cada ilha parece um bloco sólido e definido. Valores altos (0.5+) fazem com que os pontos fiquem mais distantes, o que espalha as ilhas;

* spread: define a escala de distribuição dos pontos. Spread = 1.0 mantém a proporção real da separação, evitando que os clusters se encostem. Diminuir ajuda a nitidez.

* resolução do algoritmo Leiden: diz ao algoritmo o quão agressivo ele deve ser ao fatiar a rede de vizinhos para criar os clusters. Valores baixos (0.1 a 0.5)forçam o algoritmo a criar poucos grupos, amplos e genéricos. Ideal para separar apenas as grandes massas (ex: "Células de Câncer" vs. "Células Saudáveis"). Valores altos (0.8 a 1.5) fatiam os dados em dezenas de micro-grupos. Ideal para encontrar subtipos celulares raros dentro do tumor.

Na função `neighbors` a métrica de discância selecionada afeta o agrupamento, portanto, pode ser necessário explorar diferentes métricas a depender do dataset. Métricas comuns do `Scanpy` incluem:
* Distância Euclidiana: distância em linha reta entre dois pontos. Tende a agrupar células com magnitudes de expressão geral semelhantes. Genes de expressão muito alta podem dominar o cálculo da distância;
* Distância de Manhattan(L1 distance, City Block distance): Soma das diferentes absolutas entre suas coordenadas. Menos sensível a valores discrepantes do que a euclidiana. Útil quando as diferenças em caracterísitcas individuais são mais importantes do que a magnitude geral;
* Cosine distance/similarity: Mede o ângulo entre dois vetores e um ângulo menor indica maior similaridade. Se concentra na orientação dos perfis de expressão, em vez de sua magnitude. Particularmente útil quando as porporções relativas da expressão gênica podem ser mais informativas do que as contagens absolutas. Segmentações celulares que expressam os mesmos genes em proporções semelhantes serão consideradas próximas, mesmo que uma delas tenha uma contagem total maior;
* Correlation-based distances (e.g., Pearson, Spearman): normalmente definem a distância como `1 - correlation_coefficient`. Os bins de segmentação celular são considerados semelhantes se seus perfis de expressão gênica forem altamente correlacionados, independentemente dos valores absolutos de expressão. Este método identifica agrupamentos de segmentação celular com padrões semelhantes de atividade gênica.


Uma métrica que enfatiza a magnitude (como a distância euclidiana) pode conectar células com base na atividade transcricional geral, enquanto uma métrica que enfatiza a forma (como o cosseno) pode conectar células com padrões de expressão gênica semelhantes, mesmo que seu conteúdo total de RNA seja diferente.

Para esta análise, utilizamos a métrica de distância de correlação do `Scanpy`, com resolução de agrupamento (RES) definida como 0,8 e número de vizinhos como 15. Esses parâmetros provavelmente precisarão ser ajustados para novas análises: uma resolução menor geralmente leva a menos agrupamentos, enquanto o aumento do número de vizinhos terá um efeito semelhante.



In [ ]:
%%time

#configurações iniciais
RES = 0.5      #resolução do algoritmo Leiden
NEIGHBORS = 30 #número de vizinhos
MIN_DIST = 0.5 #distância mínima
SPREAD = 2.0   #espalhamento

#cálculo de vizinhos - gera matriz de distância entre um vizinho e um grafo
sc.pp.neighbors(adata, n_neighbors=NEIGHBORS, use_rep="X_pca", metric="correlation")

# clustering Leiden
sc.tl.leiden(adata, flavor="igraph", key_added="clusters", resolution=RES, random_state=0)

#reordenação dos clusters
adata.obs['orig_clusters'] = adata.obs['clusters']
clusters = adata.obs['clusters'].astype(int)
cluster_order = {old: str(new) for new, old in enumerate(clusters.value_counts().index)}
adata.obs['clusters'] = clusters.map(cluster_order).astype('category')

#UMAP
sc.tl.umap(adata, min_dist=MIN_DIST, spread=SPREAD, random_state=0)

#checkpoint
adata.write(os.path.join(path_crc, 'adata_100k_clustered.h5ad'))

In [ ]:
#visualização
#puxa do drive se o objeto não estiver na memória
if 'adata' not in locals():
    import scanpy as sc
    path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'
    adata = sc.read_h5ad(os.path.join(path_crc, 'adata_100k_clustered.h5ad'))

#mapeamento para ajustar os nomes dos pacientes na legenda
mapeamento = {"Cancer_P1": "P1", "Cancer_P2": "P2", "Normal_P3": "P3", "Normal_P5": "P5"}
adata.obs['sample_clean'] = adata.obs['sample'].map(mapeamento)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

#clusters
sc.pl.umap(adata, color="clusters", title="Clusters Identificados",
            ax=ax1, show=False, size=2, palette="tab20")
ax1.set(xlabel="UMAP 1", ylabel="UMAP 2")
ax1.set_box_aspect(0.8) #trava o tamanho

#amostras
sc.pl.umap(adata, color="sample_clean", title="Distribuição por Amostra",
            ax=ax2, show=False, size=2)
ax2.set(xlabel="UMAP 1", ylabel="UMAP 2")
ax2.set_box_aspect(0.8) #trava o tamanho
ax2.legend(title="Amostra", bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False) #legenda

plt.tight_layout()
plt.show()

#heatmap
ct = pd.crosstab(adata.obs["sample"], adata.obs["clusters"], normalize='index') * 100
ct.index = ct.index.map(mapeamento) #usando o mesmo mapeamento definido acima
plt.figure(figsize=(12, 4))
sns.heatmap(ct, annot=True, fmt=".1f", cmap="YlOrRd")
plt.title("Composição percentual dos clusters por amostra")
plt.ylabel("Amostra")
plt.xlabel("Cluster")
plt.show()

Resultados:
* clusters bastante espalhados;
* amostras muito isoladas, o que indica efeito de lote;
* heatmap confirma efeito de lote, o clustering não enxerga as células e sim os pacientes de origem.

## **4.2. Amostra de 100k + HVG**


### 4.2.1. PCA


In [ ]:
path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'

#verifica se o objeto está na memória, se não, carrega do drive
if 'concatenated_sdata' not in locals():
    import spatialdata as spd
    concatenated_sdata = spd.read_zarr(os.path.join(path_crc, 'sdata_100k_raw.zarr'))

#recupera o dataset completo filtrado para 650k
adata = concatenated_sdata.tables['segmentation_counts'].copy()

#reaplica filtros básicos para garantir consistência
sc.pp.filter_genes(adata, min_cells=50)
min_counts = np.expm1(4).astype("int")
max_counts = np.expm1(8).astype("int")
sc.pp.filter_cells(adata, min_counts=min_counts)
sc.pp.filter_cells(adata, max_counts=max_counts)

#processamento estatístico
sc.pp.normalize_total(adata, target_sum=None)
sc.pp.log1p(adata)

#HVG
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat", batch_key="sample")

#PCA focado apenas nos genes variáveis
sc.tl.pca(adata, svd_solver='arpack', use_highly_variable=True)

#checkpoint
adata.write_h5ad(os.path.join(path_crc, 'adata_100k_pre_sketch.h5ad'))

gc.collect()

### 4.2.2. Geosketch

In [ ]:
#geosketch para amostra de 100k
N_SAMPLES = 100000
sketch_index = sketch.gs(adata.obsm['X_pca'], N_SAMPLES, replace=False)
adata = adata[sketch_index].copy() #cria novo objeto amostrado e libera o anterior da memória
gc.collect()

### 4.2.3. Clustering e UMAP

In [ ]:
%%time

#se não estiver na memória busca no drive
if 'adata' not in locals():
    import scanpy as sc
    path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'
    adata = sc.read_h5ad(os.path.join(path_crc, 'adata_100k_sketched.h5ad'))

#hiperparâmetros
RES = 0.5; NEIGHBORS = 30; MIN_DIST = 0.5; SPREAD = 2.0

#cálculo de vizinhos e clustering Leiden
sc.pp.neighbors(adata, n_neighbors=NEIGHBORS, use_rep="X_pca", metric="correlation")
sc.tl.leiden(adata, flavor="igraph", key_added="clusters", resolution=RES, random_state=0)

#UMAP
sc.tl.umap(adata, min_dist=MIN_DIST, spread=SPREAD, random_state=0)

#mapeamento para ajustar os nomes dos pacientes na legenda
mapeamento = {"Cancer_P1": "P1", "Cancer_P2": "P2", "Normal_P3": "P3", "Normal_P5": "P5"}
adata.obs['sample_clean'] = adata.obs['sample'].map(mapeamento)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

#clusters
sc.pl.umap(adata, color="clusters", title="Clusters Identificados (100k + HVG)",
           ax=ax1, show=False, size=2, palette="tab20")
ax1.set(xlabel="UMAP 1", ylabel="UMAP 2")
ax1.set_box_aspect(0.8)

#amostras
sc.pl.umap(adata, color="sample_clean", title="Distribuição por Amostra (100k + HVG)",
           ax=ax2, show=False, size=2)
ax2.set(xlabel="UMAP 1", ylabel="UMAP 2")
ax2.set_box_aspect(0.8)
ax2.legend(title="Amostra", bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)

plt.tight_layout()
plt.show()

#heatmap
ct = pd.crosstab(adata.obs["sample"], adata.obs["clusters"], normalize='index') * 100
ct.index = ct.index.map(mapeamento)

plt.figure(figsize=(12, 4))
sns.heatmap(ct, annot=True, fmt=".1f", cmap="YlOrRd")
plt.title("Composição Percentual dos Clusters por Amostra (100k + HVG)")
plt.ylabel("Amostra")
plt.xlabel("Cluster")
plt.show()

gc.collect()

#checkpoint clustering e umap
path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'
adata.write_h5ad(os.path.join(path_crc, 'adata_100k_hvg_umap.h5ad'))

Resultados:
* geosketch: 7 minutos; clustering e umap: 1:42s
* após a utilização do HVG o número de clusters encontrado foi menor. Sem o HVG, o PCA tenta explicar a variação de todos os genes, contando com muita contaminação ambiental. O Leiden enxergava como se fosse biologia e criava micro-grupos que não existem de fato.
* ao selecionar os 2k mais importantes o algoritmo enxerga estruturas mais sólidas, o que funde esses grupos em clusters maiores e mais coesos.
* batch_key="sample" força o scanpy a procurar genes que variam muito dentro de cada amostra, o que prioriza a biologia real. Sem ele, o código pega genes que variam muito no dataset como um todo, que eram os dos P1 e P2. Isso faz com que surjam menos clusters, mas que livres do efeito de lote representam tipos celulares e não pacientes.
* para obter mais clusters o caminho não é mexer no HVG, sim em resolução.
* P2 ainda parece uma ilha isolada do restante;
* o heatmap parece um pouco melhor no geral, mas ainda tem clusters muito específicos de um paciente só;
* HVG não foi suficiente para corrigir o efeito de lote, o que indica necessidade de integração.

## **4.3. Amostra de 100k + HVG + Harmony**



O algoritmo Harmony (Korsunsky et al., 2019) opera via projeção iterativa das observações em um espaço compartilhado, ajustando as coordenadas do PCA para mitigar discrepâncias entre diferentes pacientes ou corridas de sequenciamento. Por atuar diretamente no ajuste dos componentes principais, esta etapa foi executada após o PCA e antes da construção do grafo de vizinhança, seguindo as recomendações técnicas do pacote (Scanpy, 2026b).

Usa o adata do 4.2.2.

####4.3.1. Default

In [ ]:
%%time

#se não estiver na memória busca no drive
if 'adata' not in locals():
    import scanpy as sc
    path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'
    adata = sc.read_h5ad(os.path.join(path_crc, 'adata_100k_hvg_umap.h5ad'))

# neighborhood and clustering resolution
RES = 0.5       # clustering resolution
NEIGHBORS = 30  # number of neighbors
MIN_DIST=0.5    #default 0.5
SPREAD=2        #default 1

# Performing batch correction
adata_harmony = adata.copy()
import scanpy.external as sce
sce.pp.harmony_integrate(adata_harmony, key="sample", basis="X_pca", max_iter_harmony=20)

#substitui o PCA original pelo PCA corrigido pelo Harmony para os cálculos seguintes
adata_harmony.obsm["X_pca_orig"] = adata_harmony.obsm["X_pca"].copy() # backup do original
adata_harmony.obsm["X_pca"] = adata_harmony.obsm["X_pca_harmony"]

#recalcula vizinhos, clusters e umap no novo espaço integrado
sc.pp.neighbors(adata_harmony, n_neighbors=NEIGHBORS, use_rep="X_pca", metric="correlation")
sc.tl.leiden(adata_harmony, flavor="igraph", key_added="clusters", resolution=RES, random_state=0)
sc.tl.umap(adata_harmony, min_dist=MIN_DIST, spread=SPREAD, random_state=0)

#visualização
mapeamento = {"Cancer_P1": "P1", "Cancer_P2": "P2", "Normal_P3": "P3", "Normal_P5": "P5"}
adata_harmony.obs['sample_clean'] = adata_harmony.obs['sample'].map(mapeamento)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

#clusters
sc.pl.umap(adata_harmony, color="clusters", title="Clusters Identificados (Harmony default1)",
           ax=ax1, show=False, size=2, palette="tab20")
ax1.set(xlabel="UMAP 1", ylabel="UMAP 2")
ax1.set_box_aspect(0.8)

#amostras
sc.pl.umap(adata_harmony, color="sample_clean", title="Distribuição por Amostra (Harmony default1)",
           ax=ax2, show=False, size=2)
ax2.set(xlabel="UMAP 1", ylabel="UMAP 2")
ax2.set_box_aspect(0.8)
ax2.legend(title="Amostra", bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)

plt.tight_layout()
plt.show()

#heatmap
ct = pd.crosstab(adata_harmony.obs["sample"], adata_harmony.obs["clusters"], normalize='index') * 100
ct.index = ct.index.map(mapeamento)

plt.figure(figsize=(12, 4))
sns.heatmap(ct, annot=True, fmt=".1f", cmap="YlOrRd")
plt.title("Composição Percentual dos Clusters por Amostra (Harmony default1)")
plt.ylabel("Amostra")
plt.xlabel("Cluster")
plt.show()

# Atualiza o objeto principal com a versão integrada antes de salvar
adata = adata_harmony
del adata_harmony
gc.collect()

#checkpoint
path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'
adata.uns['experiment_note'] = "Harmony default"
adata.write_h5ad(os.path.join(path_crc, 'adata_100k_hvg_harmony_default.h5ad'))

Resultados:
* 2:15s
* Harmony convergiu em 2 iterações;
* com exceção do P2, os pacientes estão muito mais misturados;
* heatmap mostra clusters com a composição melhor distribuída entre os pacientes;
* a integração com Harmony foi bem sucedida, conseguiu alinhar populações celulares parecidas.

####4.3.2. Parâmetros ajustados

In [ ]:
%%time

#recuperação
path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'
if 'adata' not in locals():
    import scanpy as sc
    adata = sc.read_h5ad(os.path.join(path_crc, 'adata_100k_hvg_umap.h5ad'))

if 'concatenated_sdata' not in locals():
    import spatialdata as spd
    concatenated_sdata = spd.read_zarr(os.path.join(path_crc, 'sdata_100k_raw.zarr'))

# neighborhood and clustering resolution - PARÂMETROS AJUSTADOS
RES = 1.2        #aumenta a resolução para forçar a detecção de grupos menores
NEIGHBORS = 10   #reduz para o UMAP focar apenas no que é muito parecido
MIN_DIST = 0.05  #reduz para forçar os grupos a se separarem em ilhas
SPREAD = 1.5

# Performing batch correction
#usando 'adata' (100k+HVG) como base
adata_harmony = adata.copy()
import scanpy.external as sce
sce.pp.harmony_integrate(adata_harmony, key="sample", basis="X_pca", max_iter_harmony=20)

# Copying the harmony PCA embedding results
adata_harmony.obsm["X_pca_orig"] = adata_harmony.obsm["X_pca"]
adata_harmony.obsm["X_pca"] = adata_harmony.obsm["X_pca_harmony"]
adata_harmony.obs["cluster_orig"] = adata_harmony.obs["clusters"]

sc.pp.neighbors(adata_harmony, n_neighbors=NEIGHBORS, use_rep="X_pca", metric="correlation")
sc.tl.leiden(adata_harmony, flavor="igraph", key_added="harmony_clusters", resolution=RES, random_state=0)
adata_harmony.obs["clusters"] = adata_harmony.obs["harmony_clusters"]

# Set random_state for reproducible UMAP
sc.tl.umap(adata_harmony, min_dist=MIN_DIST, spread=SPREAD, random_state=0)

#visualização
mapeamento = {"Cancer_P1": "P1", "Cancer_P2": "P2", "Normal_P3": "P3", "Normal_P5": "P5"}
adata_harmony.obs['sample_clean'] = adata_harmony.obs['sample'].map(mapeamento)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

#clusters
sc.pl.umap(adata_harmony, color="clusters", title="Clusters Identificados (100k + HVG + Harmony ajustado)",
           ax=ax1, show=False, size=2, palette="tab20")
ax1.set(xlabel="UMAP 1", ylabel="UMAP 2")
ax1.set_box_aspect(0.8)

#amostras
sc.pl.umap(adata_harmony, color="sample_clean", title="Distribuição por Amostra (100k + HVG + Harmony ajustado)",
           ax=ax2, show=False, size=2)
ax2.set(xlabel="UMAP 1", ylabel="UMAP 2")
ax2.set_box_aspect(0.8)
ax2.legend(title="Amostra", bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)

plt.tight_layout()
plt.show()

#heatmap
ct = pd.crosstab(adata_harmony.obs["sample"], adata_harmony.obs["clusters"], normalize='index') * 100
ct.index = ct.index.map(mapeamento)

plt.figure(figsize=(12, 4))
sns.heatmap(ct, annot=True, fmt=".1f", cmap="YlOrRd")
plt.title("Composição Percentual dos Clusters por Amostra (100k + HVG + Harmony ajustado)")
plt.ylabel("Amostra")
plt.xlabel("Cluster")
plt.show()

#sobrescreve
adata = adata_harmony
concatenated_sdata.tables['segmentation_counts'] = adata

# Limpa RAM
del adata_harmony, RES, NEIGHBORS, MIN_DIST, SPREAD
gc.collect()

#checkpoint final da seção no Drive
adata.write_h5ad(os.path.join(path_crc, 'adata_100k_hvg_harmony.h5ad'))
concatenated_sdata.write(os.path.join(path_crc, 'sdata_100k_hvg_harmony.zarr'), overwrite=True)

print("---------------------------------")

Resultados:
* Harmony convergiu em 2 iterações;
* aumento drástico no número de clusters;
* umap composto por ilhas pequenas e bem separadas, efeito do ajuste feito no min_dist, que força a separação máxima e a alta resolução do Leiden fragmenta os clusters;
* os pacientes estão mais misturados;
* os ajustes trouxeram maior sensibilidade, o que é bom para identificar sub-populações que o geosketch normal poderia perder;
* o heatmap apresenta uma possível divisão excessiva, o que dificulta a interpretação.

## **4.4. Amostra de 200k + HVG + Harmony**

Teste com 200k bins para encontrar o equilíbrio ideal.
Objetivo: entender se há ganhos quando é usado o dobro de dados.



###4.4.1. PCA

In [ ]:
# reset para o dataset completo de 650k do drive
path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'
concatenated_sdata = spd.read_zarr(os.path.join(path_crc, 'sdata_100k_raw.zarr')) #força leitura do arquivo bruto
adata = concatenated_sdata.tables['segmentation_counts'].copy()

#filtros de consistência
sc.pp.filter_genes(adata, min_cells=50)
min_counts, max_counts = np.expm1(4).astype("int"), np.expm1(8).astype("int")
sc.pp.filter_cells(adata, min_counts=min_counts)
sc.pp.filter_cells(adata, max_counts=max_counts)

#processamento e HVG
sc.pp.normalize_total(adata, target_sum=None)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat", batch_key="sample")

# PCA HVG
sc.tl.pca(adata, svd_solver='arpack', use_highly_variable=True)

# gráfico de cotovelo
plt.figure(figsize=(12, 6))
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=50)
plt.show()

gc.collect()

# checkpoint
adata.write_h5ad(os.path.join(path_crc, 'adata_200k_raw_processed.h5ad'))
gc.collect()

###4.4.2. Geosketch

In [ ]:
#amostragem estratégica expandida
N_SAMPLES = 200000
sketch_index = sketch.gs(adata.obsm['X_pca'], N_SAMPLES, replace=False)

adata = adata[sketch_index].copy()
gc.collect()

###4.4.3. Clustering e UMAP

In [ ]:
%%time

# recuperação - garante que os 200k estão na memória
path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'
if 'adata' not in locals():
    adata = sc.read_h5ad(os.path.join(path_crc, 'adata_200k_raw_processed.h5ad'))

if 'concatenated_sdata' not in locals():
    concatenated_sdata = spd.read_zarr(os.path.join(path_crc, 'sdata_100k_raw.zarr'))

# parâmetros ajustados
RES = 1.2; NEIGHBORS = 10; MIN_DIST = 0.05; SPREAD = 1.5

# Batch Correction
adata_harmony = adata.copy()
import scanpy.external as sce
sce.pp.harmony_integrate(adata_harmony, key="sample", basis="X_pca", max_iter_harmony=20)

# Atualização do espaço latente
adata_harmony.obsm["X_pca_orig"] = adata_harmony.obsm["X_pca"]
adata_harmony.obsm["X_pca"] = adata_harmony.obsm["X_pca_harmony"]

#clustering e UMAP
sc.pp.neighbors(adata_harmony, n_neighbors=NEIGHBORS, use_rep="X_pca", metric="correlation")
sc.tl.leiden(adata_harmony, flavor="igraph", key_added="harmony_clusters", resolution=RES, random_state=0)
adata_harmony.obs["clusters"] = adata_harmony.obs["harmony_clusters"]
sc.tl.umap(adata_harmony, min_dist=MIN_DIST, spread=SPREAD, random_state=0)

#visualização
mapeamento = {"Cancer_P1": "P1", "Cancer_P2": "P2", "Normal_P3": "P3", "Normal_P5": "P5"}
adata_harmony.obs['sample_clean'] = adata_harmony.obs['sample'].map(mapeamento)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

#clusters
sc.pl.umap(adata_harmony, color="clusters", title="Clusters Identificados (200k + HVG + Harmony)",
           ax=ax1, show=False, size=2, palette="tab20")
ax1.set(xlabel="UMAP 1", ylabel="UMAP 2")
ax1.set_box_aspect(0.8)

#amostras
sc.pl.umap(adata_harmony, color="sample_clean", title="Distribuição por Amostra (200k + HVG + Harmony)",
           ax=ax2, show=False, size=2)
ax2.set(xlabel="UMAP 1", ylabel="UMAP 2")
ax2.set_box_aspect(0.8)
ax2.legend(title="Amostra", bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False)

plt.tight_layout()
plt.show()

#heatmap
ct = pd.crosstab(adata_harmony.obs["sample"], adata_harmony.obs["clusters"], normalize='index') * 100
ct.index = ct.index.map(mapeamento)

plt.figure(figsize=(12, 4))
sns.heatmap(ct, annot=True, fmt=".1f", cmap="YlOrRd")
plt.title("Composição Percentual dos Clusters por Amostra (200k + HVG + Harmony)")
plt.ylabel("Amostra")
plt.xlabel("Cluster")
plt.show()

#atualiza o objeto principal e sincroniza a tabela no SpatialData
adata = adata_harmony
concatenated_sdata.tables['segmentation_counts'] = adata

#limpa RAM
del adata_harmony
gc.collect()

#checkpoint
adata.write_h5ad(os.path.join(path_crc, 'adata_200k_hvg_harmony.h5ad'))
concatenated_sdata.write(os.path.join(path_crc, 'sdata_200k_hvg_harmony.zarr'), overwrite=True)

print("---------------------------------")

Resultados:

* geosketch: 23min
* clustering e umap:
* iterações do Harmony: 2
* foi viável escalar o processo no ambiente Colab Pro, a memória RAM suportou o processamento;
* o aumento da carga de dados aumentou consideravelmente o tempo de processamento;
* os clusters são mais densos e bem definidos;
* os pacientes parecem mais misturados;
* o heatmap apresenta uma distribuição muito melhor, não apresentando tantos zeros;
* o aumento da resolução para 200k permitiu que o Harmony trabalhasse com mais dados de vizinhança, gerando uma integração mais robusta e suave. Isso prova que para o Visium HD o volume de dados importa, pois dobrar a amostra não só melhorou a sensibilidade (gerando mais clusters) mas também melhorou a qualidade da integração.

Final:
* o pipeline default não foi efetivo
* Harmony foi essencial
* Visium HD melhora com amostragem maior.

### **4.4.4. Visualização espacial**

Anotação dos clusters obtidos anteriormente. Visualização dos resultados do agrupamento diretamente nas imagens de microscopia com `render_image` e `render_shape` do objeto SpatialData.

Para que apenas parte relevante da imagem seja utilizada, aplica-se a função auxiliar crop0. A caixa delimitadora (bbox) para a área associada ao bin é criada com a função `get_extend` do SpatialData, que fornece um dicionário com coordenadas x e y e mínimas e máximas.

Esse corte é importante porque geralmente o microscópio capta uma área maior que o CytAssist.



In [ ]:
#recupera objeto 200k + HVG + Harmony do Drive
path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'
if 'concatenated_sdata' not in locals():
    concatenated_sdata = spd.read_zarr(os.path.join(path_crc, 'sdata_200k_hvg_harmony.zarr'))

image_elements = list(concatenated_sdata.images.keys())
shape_elements = list(concatenated_sdata.shapes.keys())

# We are going to create a bounding box to crop the data to the capture area.
extents = []

for i in range(len(image_elements)):
    extent = spd.get_extent(concatenated_sdata,elements=[shape_elements[i]],coordinate_system='downscale_to_hires')
    extents.append(extent)

# Plotting
if len(image_elements) != len(shape_elements):
    print("Check the spatial data to make sure that for every image there is a shape")
else:
    for i in range(len(image_elements)):
        print("Plotting: "+ image_elements[i])
        title = image_elements[i].replace("_hires_tissue_image","")
        crop0(concatenated_sdata,crs="downscale_to_hires",bbox=extents[i]).pl.render_images(image_elements[i]).pl.render_shapes(shape_elements[i],color="clusters").pl.show(coordinate_systems="downscale_to_hires", title=title)

### **4.4.5. Genes marcadores**

Genes marcadores são o que a literatura define que deveria ser encontrado. Se o pipeline estiver bem ajustado, os genes da lista devem aparecer no topo do ranking estatístico.

Aqui tem duas formas de validar:
* marcadores canônicos, tirados da literatura, que confirmam se o cluster é o que se suspeita. É mais rápido e seguro biologicamente, mas pode ser que um gene simplesmente esteja pouco expresso naquela amostra;
* data-driven, estatística sobre a amostra. Serve para descobrir novos marcadores ou subtipos, encontra genes específicos no caso de um tumor, como assinaturas de prognóstico. Pode ser que ranqueie genes de ruído se não forem filtrados.

`rank_genes_groups` do Scanpy identifica genes marcadores de cada cluster. Os resultados podem ser classificados por pontuação do marcador ou pela variação logarítmica da expressão gênica. Os genes com melhor classificação em cada cluster podem então ser analisados posteriormente com ferramentas como Enrichr para inferir o tipo celular potencial do cluster.

Estratégia adotada:
1. Executar o ranking para o algoritmo identificar os genes que definem cada grupo, comparando cada cluster contra o resto do tecido. Ele identifica os genes estatisticamente excluisivos de cada grupo
2. consultar na literatura os top genes de cada cluster e procurar nas tabelas da literatura
3. confirma o match, se os genes achados pelo código baterem com a lista que o artigo define como um determinado grupo, tem a prova para rotular o cluster.

Classificação por:
* score: representa confiança estatística, quanto maior o score, mais o algoritmo tem certeza de que aquele gene é diferente naquele cluster;
* logfoldchanges(LFC): representa magnitude da diferença, é mais útil para identificar o nome da célula. Um gene ocm LFC alto significa que ele está muito mais expresso naquele cluster do que no restante do tecido, o que o torna um marcador biológico melhor.

A anotação dos clusters foi realizada através de uma abordagem Data-Driven de Validação Cruzada. Primeiro, identificamos os genes marcadores via teste de Wilcoxon; em seguida, cruzamos esses achados com as assinaturas de referência de Oliveira et al. (2025). Outra abordagem é a de marcadores canônicos, que não foi aplicada aqui mas fica o código para aplicações futuras:



```
# cannonical markers for annotation
marker_genes = {
    "Fibroblasts": ["COL1A1", "MMP2", 'VIM'],
    "Goblet cells": ["FCGBP", "MUC2", "CLCA1"],
    "Enterocyte":["EPCAM","KRT8","KRT20","FABP2"],
    "Plasma B cells":['JCHAIN','IGKC','IGHG1'],
    "B cell":['MS4A1','CD74','CD19','CD22'],
    "Smooth muscle":['TAGLN','DES','MYH11'],
    "Tumor":["CEACAM6",'REG1B',"REG1A"]
}

# Plot dotplot for initial cluster assessment

sc.pl.dotplot(adata = concatenated_sdata["segmentation_counts"], var_names = marker_genes, groupby="clusters", standard_scale="var")

```


In [ ]:
%%time

#recuperação - objeto final de 200k na memória
path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'
if 'concatenated_sdata' not in locals():
    import spatialdata as spd
    concatenated_sdata = spd.read_zarr(os.path.join(path_crc, 'sdata_200k_hvg_harmony.zarr'))

# Obtain cluster-specific marker genes
sc.tl.rank_genes_groups(adata = concatenated_sdata["segmentation_counts"], groupby="clusters", method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(adata = concatenated_sdata["segmentation_counts"], groupby="clusters", standard_scale="var", n_genes=5)
df_marker_genes = sc.get.rank_genes_groups_df(adata = concatenated_sdata["segmentation_counts"], group = None, pval_cutoff=0.05)

# AJUSTE: Ordenação por cluster e magnitude da variação (logfoldchanges)
# O logfoldchanges é a métrica preferencial para identificar a identidade do cluster (marcador biológico),
# enquanto o score reflete a confiança estatística do algoritmo.
df_marker_genes = df_marker_genes.sort_values(by=['group', 'logfoldchanges'], ascending=[True, False])

#checkpoint
# Adicionado index=False para evitar colunas redundantes no CSV final
df_marker_genes.to_csv(os.path.join(path_crc, "marker_genes_200k_pval.csv"), index=False)

print(f"Tabela de marcadores salva em: {path_crc}")

In [ ]:
#puxa o arquivo que acabou de salvar
path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'
file_path = os.path.join(path_crc, "marker_genes_200k_pval.csv")

#le marcadores calculados pelo pipeline
df_marcadores = pd.read_csv(file_path)

#top5 genes por cluster - o arquivo já está ordenado
top5_por_cluster = df_marcadores.groupby('group').head(5)

#linha = cluster; coluna = genes
resumo_traducao = top5_por_cluster.groupby('group')['names'].apply(lambda x: ', '.join(x)).reset_index()
resumo_traducao.columns = ['Cluster ID', 'Top 5 Genes']

display(resumo_traducao)

#checkpoint
resumo_traducao.to_csv(os.path.join(path_crc, "top5_genes_por_cluster_rank.csv"), index=False)

####**4.4.5.1. Comparação com a literatura**
Arquivos disponíveis na seção "Extended data" do [artigo de referência](https://www.nature.com/articles/s41588-025-02193-3#Sec27).


In [ ]:
path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'
path_tables = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/tables'
os.makedirs(path_tables, exist_ok=True)

file_meus_marcadores = os.path.join(path_crc, 'marker_genes_200k_pval.csv')

#dicionário de arquivos da literatura
urls_nature = {
    "fig2_probes": "https://static-content.springer.com/esm/art%3A10.1038%2Fs41588-025-02193-3/MediaObjects/41588_2025_2193_MOESM7_ESM.csv",
    "fig4_proportions": "https://static-content.springer.com/esm/art%3A10.1038%2Fs41588-025-02193-3/MediaObjects/41588_2025_2193_MOESM8_ESM.csv",
    "fig5_markers_enrichment": "https://static-content.springer.com/esm/art%3A10.1038%2Fs41588-025-02193-3/MediaObjects/41588_2025_2193_MOESM9_ESM.xlsx",
    "fig6_bins_deconv": "https://static-content.springer.com/esm/art%3A10.1038%2Fs41588-025-02193-3/MediaObjects/41588_2025_2193_MOESM10_ESM.csv",
    "ext_fig3_cell_counts": "https://static-content.springer.com/esm/art%3A10.1038%2Fs41588-025-02193-3/MediaObjects/41588_2025_2193_MOESM12_ESM.csv",
    "ext_fig4_bins_type": "https://static-content.springer.com/esm/art%3A10.1038%2Fs41588-025-02193-3/MediaObjects/41588_2025_2193_MOESM13_ESM.csv",
    "ext_fig7_global_atlas": "https://static-content.springer.com/esm/art%3A10.1038%2Fs41588-025-02193-3/MediaObjects/41588_2025_2193_MOESM15_ESM.xlsx",
    "ext_fig8_deconv_tcells": "https://static-content.springer.com/esm/art%3A10.1038%2Fs41588-025-02193-3/MediaObjects/41588_2025_2193_MOESM16_ESM.csv",
    "ext_fig10_raw_umi": "https://static-content.springer.com/esm/art%3A10.1038%2Fs41588-025-02193-3/MediaObjects/41588_2025_2193_MOESM17_ESM.csv"
}

#downloads
for name, url in urls_nature.items():
    extension = ".xlsx" if "xlsx" in url else ".csv"
    filename = f"{name}{extension}"
    if not os.path.exists(filename):
        with open(filename, 'wb') as f:
            f.write(requests.get(url).content)
        print(f"✓ {filename} baixado.")
    else:
        print(f"• {filename} já disponível.")

#####**4.4.5.1.1. Source Data Fig. 5**

Referência:
* arquivo: Source Data Fig. 5: Enrichment analysis results, per-cluster gene expression values and cell–cell communication analysis results.
* figura: Fig. 5: Identification and localization of two macrophage subpopulations in the TME. c, Dot plot showing expression profiles of goblet cell subpopulations identified via unsupervised clustering

Objetivos: validação por cruzamento de dados. Pega os top 50 genes identificados para cada cluster e compara com a lista publicada no artigo para verificar quantos batem.

Procedimento:
* rank_genes_groups na etapa anterior separou o top 50 genes por clusters;
* esses genes foram comparados com o dataset usado para criação de um gráfico de dispersão mostrando os perfis de expressão das subpopulações de células caliciformes (Goblet) identificadas por meio de agrupamento não supervisionado.

In [ ]:
#garante drive acessível
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'
file_meus_marcadores = os.path.join(path_crc, 'marker_genes_200k_pval.csv')

#carrega marcadores
df_meu = pd.read_csv(file_meus_marcadores)
df_meu['names'] = df_meu['names'].str.upper().str.strip()

#seleciona a aba '5C'
df_nat_5 = pd.read_excel('fig5_markers_enrichment.xlsx', sheet_name='5C')

# padroniza nomes de colunas do arquivo
df_nat_5.columns = [str(c).strip().lower() for c in df_nat_5.columns]

#identifica colunas baseadas no padrão da literatura (features.plot e id)
gene_col_nat = 'features.plot' if 'features.plot' in df_nat_5.columns else 'gene'
cluster_col_nat = 'id' if 'id' in df_nat_5.columns else 'cluster'

resultados_fig5 = []
N_UNIVERSO = 18000  #universo de genes

for meu_id in df_meu['group'].unique():
    meus_genes = set(df_meu[df_meu['group'] == meu_id].head(50)['names'])
    n_meus = len(meus_genes) # Geralmente 50

    for nat_id in df_nat_5[cluster_col_nat].unique():
        #filtra e limpa os genes da literatura
        genes_nat = set(df_nat_5[df_nat_5[cluster_col_nat] == nat_id][gene_col_nat].dropna().astype(str).str.upper().str.strip())
        K_literatura = len(genes_nat)

        intersecao = meus_genes.intersection(genes_nat)
        k_sucessos = len(intersecao)

        if k_sucessos > 0:
            #calcula p-value
            p_value = hypergeom.sf(k_sucessos - 1, N_UNIVERSO, K_literatura, n_meus)

            resultados_fig5.append({
                'Cluster': meu_id,
                'Literatura': nat_id,
                'Correspondências': k_sucessos,
                'P-Value': p_value, #significância
                'Genes': ", ".join(intersecao)
            })

df_val_fig5 = pd.DataFrame(resultados_fig5).sort_values(['Cluster', 'Correspondências'], ascending=[True, False])
print("Validação com Fig5")
display(df_val_fig5.head(10))

#export
export_folder = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/tables/crc'
export_file_path = os.path.join(export_folder, 'Source Data Fig. 5 - 200k.csv')

df_val_fig5.to_csv(export_file_path, index=False)

Resultados:

* 8 correspondências de 50 genes principais num universo de 18k genes;
* cluster 0: os valores de p-value na casa de 10e-20 provam a precisão na identificação de células calciformes;
* cluster 9: ainda muito significativo, provavelmente um subtipo ou células em estado de transição. Tem algum parentesco com o cluster 0 mas pode não ser a mesma população principal do mesmo;
* cluster 1: ~0,02 ainda é estatisticamente significativo, mas bem abaixo dos outros clusters. apresenta tendências de expressão.



#####**4.4.5.1.2. Source Data Extended Data Fig. 7**


Source Data Extended Data Fig. 7: Statistical source data.


Objetivo: identificar músculo liso (Cluster 4), fibroblastos (Cluster 20) e células imunes (B cells/Plasma)

Extended Data Fig. 7 Analysis of macrophage subpopulations in normal and tumor regions of colon cancer sections.

a. Dot plot denoting the expression of differentially expressed genes in the 4 macrophages cluster identified via unsupervised clustering.

b. Barplot with the proportion of each macrophage cluster. Colors represent distance to the tumor.

c. Barplot with the proportion of each macrophage cluster. Colors represent the different samples.

d. Spatial organization of the 4 identified macrophage populations in the colon cancer samples. Shades of gray represent normal and tumor regions in the section and colors represent the identified unsupervised clusters. Scale bars = 1mm.

In [ ]:
#carrega df da literatura
df_nat_7 = pd.read_excel('ext_fig7_global_atlas.xlsx')
df_nat_7.columns = [str(c).strip().lower() for c in df_nat_7.columns]

gene_col_nat7 = 'features.plot' if 'features.plot' in df_nat_7.columns else 'gene'
cluster_col_nat7 = 'id' if 'id' in df_nat_7.columns else 'cluster'

resultados_fig7 = []
N_UNIVERSO = 18000 #universo de genes do dataset

for meu_id in df_meu['group'].unique():
    meus_genes = set(df_meu[df_meu['group'] == meu_id].head(50)['names'])
    n_meus = len(meus_genes)

    for nat_id in df_nat_7[cluster_col_nat7].unique():
        genes_nat = set(df_nat_7[df_nat_7[cluster_col_nat7] == nat_id][gene_col_nat7].dropna().astype(str).str.upper())
        K_literatura = len(genes_nat)

        intersecao = meus_genes.intersection(genes_nat)
        k_sucessos = len(intersecao)

        if k_sucessos > 0:
            #p-value usando a distribuição hipergeométrica
            p_val = hypergeom.sf(k_sucessos - 1, N_UNIVERSO, K_literatura, n_meus)

            resultados_fig7.append({
                'Cluster': meu_id,
                'Cluster literatura': nat_id,
                'Correspondências': k_sucessos,
                'P-Value': p_val,
                'Genes': ", ".join(intersecao)
            })

df_val_fig7 = pd.DataFrame(resultados_fig7).sort_values(['Cluster', 'Correspondências'], ascending=[True, False])
print("\nValidação com Source Data Extended Data Fig. 7")
display(df_val_fig7.head(15))

#export
export_folder = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/tables/crc'
export_file_path = os.path.join(export_folder, 'Source Data Extended Data Fig. 7 - 200k.csv')

df_val_fig7.to_csv(export_file_path, index=False)

**Ranking de significância**

In [ ]:
path_tables = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/tables/crc'
df_val_fig5_200k = pd.read_csv(os.path.join(path_tables, 'Source Data Fig. 5 - 200k.csv'))
df_val_fig7_200k = pd.read_csv(os.path.join(path_tables, 'Source Data Extended Data Fig. 7 - 200k.csv'))

#prepara tabelas para comparação
df5_comp = df_val_fig5_200k[['Cluster', 'Literatura', 'P-Value']].copy()
df5_comp.columns = ['Cluster', 'ID_Nature', 'P-Value']
df5_comp['Fonte'] = 'Fig 5'

df7_comp = df_val_fig7_200k[['Cluster', 'Cluster literatura', 'P-Value']].copy()
df7_comp.columns = ['Cluster', 'ID_Nature', 'P-Value']
df7_comp['Fonte'] = 'Fig 7'

#concat e pega o melhor p-valor
df_ranking_200k = pd.concat([df5_comp, df7_comp])
df_best_hit = (
    df_ranking_200k.sort_values("P-Value")
    .groupby("Cluster")
    .first()
    .reset_index()
)

#tabela de sintese numerica
limiar = 0.05
sintese_numerica = (
    df_best_hit[df_best_hit['P-Value'] < limiar]
    .groupby(['Fonte', 'ID_Nature'])['Cluster']
    .apply(lambda x: ', '.join(map(str, sorted(x.unique().astype(int)))))
    .reset_index()
)

print(f"Melhor p-valor por cluster (200k)")
display(sintese_numerica)

# Export para registro
sintese_numerica.to_csv(os.path.join(path_tables, 'ranking_id_nature_200k.csv'), index=False)

Resultados:
* múltiplos clusters independentes convergiram para identidades celulares únicas da literatura;
* cluster 0, fig 5 - mapeou 2 clusters (0 e 9), indicando que o pipeline foi mais sensível que a anotação da literatura para Goblet cells. Os clusters 0 e 9  obtidos compartilham de uma mesma assinatura de Goblet

###4.4.6. Marcadores nos clusters
Anotação dos clusters identificados na imagem histológica.

In [ ]:
path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'

#genes marcadores específicos da amostra de 200k
df_marker_genes_200k = pd.read_csv(os.path.join(path_crc, "marker_genes_200k_pval.csv"))
df_marker_genes_200k['group'] = df_marker_genes_200k['group'].astype(str)

#constroi biblioteca de nomes com os arquivos
biblioteca = []

#aba 5A
df_5a = pd.read_excel('fig5_markers_enrichment.xlsx', sheet_name='5A')
for _, row in df_5a.iterrows():
    genes = set(str(row['Genes']).upper().split(';'))
    biblioteca.append({'Nome': row['Cluster'], 'Genes_Ref': genes})

#aba 5EF
df_5ef = pd.read_excel('fig5_markers_enrichment.xlsx', sheet_name='5EF')
for col_nome, col_gene in [('source', 'ligand.complex'), ('target', 'receptor.complex')]:
    for nome, group in df_5ef.groupby(col_nome):
        genes = set(group[col_gene].astype(str).str.upper().unique())
        biblioteca.append({'Nome': nome, 'Genes_Ref': genes})

#cruzamento
mapeamento_final = {}

for meu_id in df_marker_genes_200k['group'].unique():
    #genes dos clusters encontrados
    meus_genes = set(df_marker_genes_200k[df_marker_genes_200k['group'] == meu_id].head(50)['names'].str.upper())

    melhor_nome = f"Nature_ID_{meu_id}" #default se não achar nada
    maior_intersecao = 0

    for entrada in biblioteca:
        intersecao = len(meus_genes.intersection(entrada['Genes_Ref']))
        if intersecao > maior_intersecao:
            maior_intersecao = intersecao
            melhor_nome = entrada['Nome']

    mapeamento_final[meu_id] = melhor_nome

#aplicação e exibiçao
print("Mapeamento 200k extraído dos arquivos 5A e 5EF:")
for k in sorted(mapeamento_final.keys(), key=int):
    print(f"'{k}' : '{v}'," if (v := mapeamento_final[k]) else "")

#aplica ao objeto
if 'concatenated_sdata' in locals():
    concatenated_sdata["segmentation_counts"].obs["grouped_clusters"] = (
        concatenated_sdata["segmentation_counts"].obs['clusters'].astype(str).map(mapeamento_final).astype('category')
    )

In [ ]:
path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'

#carrega o objeto 200k
if 'concatenated_sdata' not in locals():
    import spatialdata as spd
    concatenated_sdata = spd.read_zarr(os.path.join(path_crc, 'sdata_200k_hvg_harmony.zarr'))

#recupera os clusters do objeto processado
if 'adata' not in locals():
    import scanpy as sc
    adata = sc.read_h5ad(os.path.join(path_crc, 'adata_200k_hvg_harmony.h5ad'))

concatenated_sdata["segmentation_counts"].obs['clusters'] = adata.obs['clusters']

mapeamento_final = {
    '0' : 'Goblet', '1' : 'Enterocyte', '2' : 'Macrophage-SELENOP+',
    '3' : 'CD4 T cell', '4' : 'Fibroblast', '5' : 'Endothelial',
    '6' : 'Macrophage-SELENOP+', '7' : 'Plasma', '8' : 'Mature B',
    '9' : 'Goblet', '10' : 'Pericytes', '11' : 'CAF', '12' : 'Plasma',
    '13' : 'Myofibroblast', '14' : 'Neutrophil', '15' : 'Enteric Glial',
    '16' : 'vSM', '17' : 'vSM', '18' : 'Plasma', '19' : 'CAF', '20' : 'Plasma'
}

#aplica no objeto spatialdata
concatenated_sdata["segmentation_counts"].obs["grouped_clusters"] = (
    concatenated_sdata["segmentation_counts"].obs['clusters']
    .astype(str)
    .map(mapeamento_final)
    .astype('category')
)

#exporta objeto anotado
output_path = os.path.join(path_crc, 'sdata_200k_annotated_v2.zarr')
concatenated_sdata.write(output_path, overwrite=True)

In [ ]:
path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'

# puxa o objeto ja anotado
if 'concatenated_sdata' not in locals():
    import spatialdata as spd
    concatenated_sdata = spd.read_zarr(os.path.join(path_crc, 'sdata_200k_annotated_v2.zarr'))

image_elements = list(concatenated_sdata.images.keys())
shape_elements = list(concatenated_sdata.shapes.keys())
extents = []
for i in range(len(image_elements)):
    extent = spd.get_extent(concatenated_sdata, elements=[shape_elements[i]], coordinate_system='downscale_to_hires')
    extents.append(extent)

# gera mapas espaciais
for i in range(len(image_elements)):
    print(f"Plotting: {image_elements[i]}")
    fig, ax = plt.subplots(1, 2, figsize=(18, 8))
    title = image_elements[i].replace("_hires_tissue_image","")

    # lado esquerdo: histologia pura (H&E)
    crop0(concatenated_sdata, crs="downscale_to_hires", bbox=extents[i]).pl.render_images(image_elements[i]).pl.show(ax=ax[0], title=f"{title} - Histologia (H&E)")

    # lado direito: identificação de clusters
    crop0(concatenated_sdata, crs="downscale_to_hires", bbox=extents[i]).pl.render_images(image_elements[i], alpha=0.3).pl.render_shapes(shape_elements[i], color="grouped_clusters").pl.show(ax=ax[1], title=f"{title} - Identificação de Clusters (200k)")

    plt.tight_layout()
    plt.show()

####**4.4.6.1. Comparação com a literatura**

#####**4.4.6.1.1. Fig. 5**

Identification and localization of two macrophage subpopulations in the TME. d, Spatial plots showing the localization of the identified goblet cell subpopulations.

Objetivo: plotar apenas Goblet Cells para comparar visualmente com a literatura


In [ ]:
# lista amostras de interesse
samples_to_plot = ['P1', 'P2', 'P5']

# loop de plotagem filtrado
for i in range(len(image_elements)):
    if not any(s in image_elements[i] for s in samples_to_plot):
        continue

    print(f"Plotando Goblet Cells (200k): {image_elements[i]}")
    fig, ax = plt.subplots(1, 2, figsize=(18, 8))
    title = image_elements[i].replace("_hires_tissue_image","")

    # lado esquerdo: Histologia Pura (H&E)
    crop0(concatenated_sdata, crs="downscale_to_hires", bbox=extents[i]).pl.render_images(image_elements[i]).pl.show(ax=ax[0], title=f"{title} - H&E Original")

    # lado direito: Goblet Cells
    crop0(concatenated_sdata, crs="downscale_to_hires", bbox=extents[i])\
        .pl.render_images(image_elements[i], alpha=0.5)\
        .pl.render_shapes(
            shape_elements[i],
            color="grouped_clusters",
            groups=['Goblet'],
            palette={'Goblet': 'blue'}
        )\
        .pl.show(ax=ax[1], title=f"{title} - Goblet Cells (200k)")

    plt.tight_layout()
    plt.show()

Resultados:
* fragmentação severa do sinal biológico nos pacientes P1 e P2;
* usar a amostra reduzida pode gerar um falso positivo, a mesma análise feita posteriormente com a base full demonstrou que existe sinal nesses pacientes;
*

#####**4.4.6.1.2 Source Data Fig. 4**

Objetivo: validação da proporção de células

* é feito o cálculo de composição relativa com value_counts para buscar o total de 200k bins e calcular quanto cada cluster ocupa;
* a literatura traz várias amostras, focando em organização espacial e proporções nas maiores linhagens (tumor, stroma, imune) então usa groupby('celltype') para extrair uma média;
* foi feito teste qui-quadrado para validar se a distribuição observada segue a distribuição esperada;
* como o N é muito grande, qualquer diferença mínima geratia um p-valor menor que 0,05. Por isso, além do p-value, foi feito um resíduo padronizado. Se o resíduo estiver entre -1,96 e +1,96 a proporção é estatisticamente igual à literatura (95% de confiança) - teste de associação e análise de resíduos - o valos na tabela Z quemarca o ponto onde restam apenas 2,5% na cauda é 1,96.

Literatura:
* Source Data Fig. 4: Cell type proportion per patient and tissue region, statistical source data.

* Fig. 4: Cellular composition of the tumor periphery in each CRC section.

Extended Data Fig. 4 Proportion of cell types identified in three CRC samples.
Barplot with the proportion of each of the identified cell types after deconvolution on the three different CRC samples. Colors represent the sample.

In [ ]:
#busca sdata anotado
path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'
concatenated_sdata = spd.read_zarr(os.path.join(path_crc, 'sdata_200k_annotated_v2.zarr'))

#proporções com os nomes anotados da amostra 200k
prop_anotada = concatenated_sdata["segmentation_counts"].obs['grouped_clusters'].value_counts(normalize=True) * 100
df_prop_final = prop_anotada.reset_index()
df_prop_final.columns = ['Tipo Celular', 'Proporção pesquisa (%)']

print("Composição do tecido (Amostra 200k):")
display(df_prop_final)

In [ ]:
#carrega literatura
df_prop_nat = pd.read_csv('fig4_proportions.csv')
df_prop_nat.columns = [str(c).strip().lower() for c in df_prop_nat.columns]

depara_nature = {
    'CAF': 'Estroma/Fibroblastos', 'Myofibroblast': 'Estroma/Fibroblastos',
    'Fibroblast': 'Estroma/Fibroblastos', 'Proliferating Fibroblast': 'Estroma/Fibroblastos',
    'Vascular Fibroblast': 'Estroma/Fibroblastos',
    'Tumor I': 'Epitelial/Tumoral', 'Tumor II': 'Epitelial/Tumoral', 'Tumor III': 'Epitelial/Tumoral',
    'Tumor IV': 'Epitelial/Tumoral', 'Tumor V': 'Epitelial/Tumoral', 'Epithelial': 'Epitelial/Tumoral',
    'Enterocyte': 'Epitelial/Tumoral', 'Tuft': 'Epitelial/Tumoral',
    'Goblet': 'Células Caliciformes',
    'Macrophage': 'Outras Imunes', 'Proliferating Macrophages': 'Outras Imunes',
    'Plasma': 'Células Plasmáticas', 'Mature B': 'Células Plasmáticas',
    'Smooth Muscle': 'Músculo Liso', 'vSM': 'Músculo Liso', 'Pericytes': 'Músculo Liso'
}

df_prop_nat['categoria_pt'] = df_prop_nat['celltype'].map(depara_nature).fillna('Outras Imunes')
resumo_nat = df_prop_nat.groupby('categoria_pt')['proportion'].sum().reset_index()

total_nat = resumo_nat['proportion'].sum()
resumo_nat['proportion'] = (resumo_nat['proportion'] / total_nat) * 100

#nomes exatos gerados pelo mapeamento da amostra de 200k
depara_pesquisa = {
    'Goblet': 'Células Caliciformes',
    'Enterocyte': 'Epitelial/Tumoral',
    'CAF': 'Estroma/Fibroblastos',
    'Fibroblast': 'Estroma/Fibroblastos',
    'Myofibroblast': 'Estroma/Fibroblastos',
    'Plasma': 'Células Plasmáticas',
    'Mature B': 'Células Plasmáticas',
    'vSM': 'Músculo Liso',
    'Pericytes': 'Músculo Liso',
    'Macrophage-SELENOP+': 'Outras Imunes',
    'CD4 T cell': 'Outras Imunes',
    'Endothelial': 'Outras Imunes',
    'Neutrophil': 'Outras Imunes',
    'Enteric Glial': 'Outras Imunes'
}

#contagens brutas (usa o objeto 200k em memória)
counts_minha = concatenated_sdata["segmentation_counts"].obs['grouped_clusters'].value_counts()
df_minha = counts_minha.reset_index()
df_minha.columns = ['original', 'contagem']
df_minha['categoria_pt'] = df_minha['original'].map(depara_pesquisa).fillna('Outras Imunes')
resumo_minha = df_minha.groupby('categoria_pt')['contagem'].sum().reset_index()

#cruzamento e teste estatístico
tabela_val = pd.merge(resumo_minha, resumo_nat, on='categoria_pt', how='inner')
N_total = tabela_val['contagem'].sum()

tabela_val['esperado_count'] = (tabela_val['proportion'] / 100) * N_total
tabela_val['Pesquisa (%)'] = (tabela_val['contagem'] / N_total) * 100
tabela_val.rename(columns={'proportion': 'Literatura (%)'}, inplace=True)

# Cálculo do Resíduo Padronizado
tabela_val['Resíduo padronizado'] = (tabela_val['contagem'] - tabela_val['esperado_count']) / np.sqrt(tabela_val['esperado_count'])

tabela_val['Validação'] = tabela_val['Resíduo padronizado'].apply(
    lambda x: 'Validado' if abs(x) < 1.96 else 'Diferença significativa'
)

print("TABELA DE VALIDAÇÃO DE PROPORÇÕES (QUI-QUADRADO - AMOSTRA 200K)")
tabela_val_exp = tabela_val[['categoria_pt', 'Pesquisa (%)', 'Literatura (%)', 'Resíduo padronizado', 'Validação']]
display(tabela_val_exp)

#export
export_folder = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/tables/crc'
export_file_path = os.path.join(export_folder, 'Source Data Fig. 4 200k.csv')
tabela_val_exp.to_csv(export_file_path, index=False)

Resultados:
* Epitelial/Tumoral: 1,74% (amostra) | 44,69% (literatura), a amostra de 200k praticamente perdeu o tumor;
* Células Plasmáticas: 16,15% (amostra) | 1,96% (literatura)
* Outras Imunes: 36,74% (amostra) | 14,16% (literatura)
  * amostra fortemente enviesada para o microambiente de células imunes;
* Goblet: 3,25% (amostra) | 11,58% (literatura), conversa com o resultado visual observado na etapa anterior, demonstrando que a amostragem não conseguiu representar esse grupo.
* resíduo padronizado de 1,96 já indica que as diferenças observadas não são do acaso;
* O teste qui-quadrado revelou que a redução da base para 200.000 bins distorceu severamente a representação biológica do tecido. Isso reforça a necessidade do processamento da base full para manter a fidelidade da arquitetura.

###**4.4.7. Análise de expressão gênica diferencial**

Agora que os clusters estão anotados, vamos realizar um pseudobulking dos dados para executar a análise de expressão gênica diferencial usando o DESeq2 nessas contagens agregadas.

A técnica de pseudobulk transforma o experimento de imagem espacial num experimento de RNA-seq clássico, que é padrão ouro para a descoberta de biomarcadores. Sai de uma visão de pontos para uma visão de comportamento de grupo. Isso porque dados de célula única são esparsos e ao somar é criado um sinal robusto. É possível ver se o volume total de uma expressão de um paciente é muito diferente de outro. Aqui entram `counts` e `metadata`.

Nesta seção, o foco será o cluster de fibroblastos, com o objetivo de identificar genes que apresentam expressão diferencial entre as amostras de câncer e tecido normal adjacente.

`aggregate` do Scanpy para agrupar a tabela de objetos AnnData pelos clusters agrupados e pela amostra nos metadados do objeto AnnData.

Camada `filtered_counts` para a expressão gênica e somamos as contagens, visto que o DESeq2 requer dados de contagem brutos como entrada.

Observação: Há apenas n=2 réplicas biológicas por condição. Para maior poder estatístico, recomenda-se um número maior de réplicas. O mínimo é n=3 e alguns estudos recomendam n=5 ou mais.


####4.4.7.2. Wilcoxon

In [ ]:
table = concatenated_sdata["segmentation_counts"]

#prepara os nomes
table.obs['tissue'] = table.obs['sample'].astype(str).str.split('_').str[0].str.strip()

#verificação
print("Grupos encontrados na coluna tissue (200k):")
print(table.obs['tissue'].value_counts())

#filtra CAFs
table_caf = table[table.obs['grouped_clusters'] == 'CAF'].copy()
sc.pp.log1p(table_caf) #normalização logarítmica ajuda a comprimir a escala e reduzir a influência de bins com outliers

#Wilcoxon (cancer vs normal)
sc.tl.rank_genes_groups(
    table_caf,
    groupby='tissue',
    reference='Normal',
    method='wilcoxon',
    key_added='wilcoxon_200k'
)

df_wilcoxon = sc.get.rank_genes_groups_df(table_caf, group='Cancer', key='wilcoxon_200k')
df_wilcoxon_sig = df_wilcoxon[(df_wilcoxon['pvals_adj'] < 0.05) & (df_wilcoxon['logfoldchanges'] > 0)]

print(f"\nWilcoxon (Amostra 200k): {len(df_wilcoxon_sig)} genes significativos.")
display(df_wilcoxon_sig.head(10))

path_tables = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/tables/crc'
df_wilcoxon_sig.to_csv(os.path.join(path_tables, 'Genes_CAF_200k_Wilcoxon.csv'), index=False)

## **4.5. Base full + Harmony + HVG**
Utilizar o dataset completo é escolha que faz sentido porque no Visium HD a qualidade dos dados é muito alta. Utilizar o transcriptoma completo preserva a resolução máxima da tecnologia.


###**4.5.1. PCA**

In [ ]:
# Reset para o dataset completo
path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'

# Carrega o objeto
concatenated_sdata = spd.read_zarr(os.path.join(path_crc, 'sdata_100k_raw.zarr'))
adata = concatenated_sdata.tables['segmentation_counts'].copy()

print(f"{adata.n_obs} bins.")

# Filtros de consistência
sc.pp.filter_genes(adata, min_cells=50)
min_counts, max_counts = np.expm1(4).astype("int"), np.expm1(8).astype("int")
sc.pp.filter_cells(adata, min_counts=min_counts)
sc.pp.filter_cells(adata, max_counts=max_counts)

# Processamento e HVG
sc.pp.normalize_total(adata, target_sum=None)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat", batch_key="sample")

# PCA HVG
sc.tl.pca(adata, svd_solver='arpack', use_highly_variable=True)

# Gráfico de cotovelo
plt.figure(figsize=(12, 6))
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=50)
plt.show()

gc.collect()

# Checkpoint
adata.write_h5ad(os.path.join(path_crc, 'adata_full_raw_processed.h5ad'))
gc.collect()

###**4.5.2. Clustering, Harmony e UMAP**

In [ ]:
%%time
path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'
if 'adata' not in locals():
    adata = sc.read_h5ad(os.path.join(path_crc, 'adata_full_processed.h5ad'))

# Parâmetros para alta densidade
RES = 1.2; NEIGHBORS = 15; MIN_DIST = 0.1; SPREAD = 1.0

# Batch Correction (Harmony)
import scanpy.external as sce
sce.pp.harmony_integrate(adata, key="sample", basis="X_pca", max_iter_harmony=25)

# Atualização do espaço latente
adata.obsm["X_pca_orig"] = adata.obsm["X_pca"].copy()
adata.obsm["X_pca"] = adata.obsm["X_pca_harmony"]

# Clustering e UMAP
sc.pp.neighbors(adata, n_neighbors=NEIGHBORS, use_rep="X_pca", metric="correlation")
sc.tl.leiden(adata, flavor="igraph", key_added="harmony_clusters", resolution=RES, random_state=0)
adata.obs["clusters"] = adata.obs["harmony_clusters"]
sc.tl.umap(adata, min_dist=MIN_DIST, spread=SPREAD, random_state=0)

# Visualização (Ajuste de size para 650k pontos)
mapeamento = {"Cancer_P1": "P1", "Cancer_P2": "P2", "Normal_P3": "P3", "Normal_P5": "P5"}
adata.obs['sample_clean'] = adata.obs['sample'].map(mapeamento)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
sc.pl.umap(adata, color="clusters", title="Clusters Identificados (Full Dataset)",
           ax=ax1, show=False, size=0.5, palette="tab20")
sc.pl.umap(adata, color="sample_clean", title="Distribuição por Amostra",
           ax=ax2, show=False, size=0.5)
plt.tight_layout()
plt.show()

# Checkpoint dos objetos integrados
adata.write_h5ad(os.path.join(path_crc, 'adata_full_harmony_umap.h5ad'))
gc.collect()

Resultados:
* tempo: 36min
* Harmony: conversão em 2 iterações

###**4.5.3. Genes marcadores**

In [ ]:
%%time

#recupera
path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'
if 'adata' not in locals():
    import scanpy as sc
    adata = sc.read_h5ad(os.path.join(path_crc, 'adata_full_harmony_umap.h5ad'))

# Obtain cluster-specific marker genes
sc.tl.rank_genes_groups(adata = adata, groupby="clusters", method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(adata = adata, groupby="clusters", standard_scale="var", n_genes=5)
df_marker_genes_full = sc.get.rank_genes_groups_df(adata = adata, group = None, pval_cutoff=0.05)

#ordena por cluster e magnitude da variação
df_marker_genes_full = df_marker_genes_full.sort_values(by=['group', 'logfoldchanges'], ascending=[True, False])

#checkpoint
df_marker_genes_full.to_csv(os.path.join(path_crc, "marker_genes_full_pval.csv"), index=False)

####**4.5.3.1. Comparação com a literatura**

#####**4.5.3.1.1. Source Data Fig. 5**

In [ ]:
from scipy.stats import hypergeom

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'
file_meus_marcadores = os.path.join(path_crc, 'marker_genes_full_pval.csv')

df_meu = pd.read_csv(file_meus_marcadores)
df_meu['names'] = df_meu['names'].str.upper().str.strip()
df_meu['group'] = df_meu['group'].astype(str)

df_nat_5 = pd.read_excel('fig5_markers_enrichment.xlsx', sheet_name='5C')
df_nat_5.columns = [str(c).strip().lower() for c in df_nat_5.columns]

gene_col_nat = 'features.plot' if 'features.plot' in df_nat_5.columns else 'gene'
cluster_col_nat = 'id' if 'id' in df_nat_5.columns else 'cluster'

resultados_fig5_full = []
N_UNIVERSO = 18000

for meu_id in df_meu['group'].unique():
    meus_genes = set(df_meu[df_meu['group'] == meu_id].head(50)['names'])
    n_meus = len(meus_genes)

    for nat_id in df_nat_5[cluster_col_nat].unique():
        #extrai genes linha a linha
        genes_nat = set(df_nat_5[df_nat_5[cluster_col_nat] == nat_id][gene_col_nat].dropna().astype(str).str.upper().str.strip())

        K_literatura = len(genes_nat)
        intersecao = meus_genes.intersection(genes_nat)
        k_sucessos = len(intersecao)

        if k_sucessos > 0:
            p_value = hypergeom.sf(k_sucessos - 1, N_UNIVERSO, K_literatura, n_meus)

            resultados_fig5_full.append({
                'Cluster': meu_id,
                'Literatura': nat_id,
                'Correspondências': k_sucessos,
                'P-Value': p_value,
                'Genes': ", ".join(intersecao)
            })

df_val_fig5_full = pd.DataFrame(resultados_fig5_full).sort_values(['Cluster', 'Correspondências'], ascending=[True, False])
display(df_val_fig5_full.head(10))

export_folder = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/tables/crc'
export_file_path = os.path.join(export_folder, 'Source Data Fig. 5 Full.csv')
df_val_fig5_full.to_csv(export_file_path, index=False)

In [ ]:
#filtro de significancia
limiar = 0.05

path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/tables/crc'
df_val_fig5_full_if = os.path.join(path_crc, 'Source Data Fig. 5 Full.csv')
df_val_fig5_full = pd.read_csv(df_val_fig5_full_if)

df_sig = df_val_fig5_full[df_val_fig5_full['P-Value'] < limiar].copy()

#agrupa por nome literatura
resumo_anotacao = (
    df_sig.groupby('Literatura')['Cluster']
    .apply(lambda x: ', '.join(map(str, sorted(x.unique().astype(int)))))
    .reset_index()
)
resumo_anotacao.columns = ['Literatura', 'Clusters correspondentes']

print(f"TABELA DE SÍNTESE BIOLÓGICA (Significância p < {limiar})")
display(resumo_anotacao)

export_folder = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/tables/crc'
export_file_path = os.path.join(export_folder, 'sintese_biologica_full.csv')
resumo_anotacao.to_csv(export_file_path, index=False)

#####**4.5.3.1.2. Source Data Extended Data Fig. 7**

In [ ]:
df_nat_7 = pd.read_excel('ext_fig7_global_atlas.xlsx')
df_nat_7.columns = [str(c).strip().lower() for c in df_nat_7.columns]

gene_col_nat7 = 'features.plot' if 'features.plot' in df_nat_7.columns else 'gene'
cluster_col_nat7 = 'id' if 'id' in df_nat_7.columns else 'cluster'

resultados_fig7_full = []
N_UNIVERSO = 18000

for meu_id in df_meu['group'].unique():
    meus_genes = set(df_meu[df_meu['group'] == meu_id].head(50)['names'])
    n_meus = len(meus_genes)

    for nat_id in df_nat_7[cluster_col_nat7].unique():
        genes_nat = set(df_nat_7[df_nat_7[cluster_col_nat7] == nat_id][gene_col_nat7].dropna().astype(str).str.upper())
        K_literatura = len(genes_nat)
        intersecao = meus_genes.intersection(genes_nat)
        k_sucessos = len(intersecao)

        if k_sucessos > 0:
            p_val = hypergeom.sf(k_sucessos - 1, N_UNIVERSO, K_literatura, n_meus)
            resultados_fig7_full.append({
                'Cluster': meu_id,
                'Cluster literatura': nat_id,
                'Correspondências': k_sucessos,
                'P-Value': p_val,
                'Genes': ", ".join(intersecao)
            })

df_val_fig7_full = pd.DataFrame(resultados_fig7_full).sort_values(['Cluster', 'Correspondências'], ascending=[True, False])
display(df_val_fig7_full.head(15))

export_file_path = os.path.join(export_folder, 'Source Data Extended Data Fig. 7 Full.csv')
df_val_fig7_full.to_csv(export_file_path, index=False)

**Ranking de significância**

Como um mesmo cluster pode ter aparecido em mais de uma correspondência, eles serão rankeados com base no valor-p para determinação de qual cluster corresponde de fato aos da literatura.

In [ ]:
#prepara tabelas para comparação
df5_comp = df_val_fig5_full[['Cluster', 'Literatura', 'P-Value']].copy()
df5_comp.columns = ['Cluster', 'ID_Nature', 'P-Value']
df5_comp['Fonte'] = 'Fig 5'

df7_comp = df_val_fig7_full[['Cluster', 'Cluster literatura', 'P-Value']].copy()
df7_comp.columns = ['Cluster', 'ID_Nature', 'P-Value']
df7_comp['Fonte'] = 'Fig 7'

#concat e pega o melhor p-valor
df_ranking_full = pd.concat([df5_comp, df7_comp])
df_best_hit = (
    df_ranking_full.sort_values("P-Value")
    .groupby("Cluster")
    .first()
    .reset_index()
)

#tabela de sintese numerica
limiar = 0.05
sintese_numerica = (
    df_best_hit[df_best_hit['P-Value'] < limiar]
    .groupby(['Fonte', 'ID_Nature'])['Cluster']
    .apply(lambda x: ', '.join(map(str, sorted(x.unique().astype(int)))))
    .reset_index()
)

print(f"Melhor p-valor por cluster)")
display(sintese_numerica)

# Export para registro
sintese_numerica.to_csv(os.path.join(export_folder, 'ranking_id_nature_full.csv'), index=False)

###**4.5.4. Marcadores nos clusters**

In [ ]:
#mapeamento dos arquivos
path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'

#carrega genes marcadores
df_marker_genes_full = pd.read_csv(os.path.join(path_crc, "marker_genes_full_pval.csv"))
df_marker_genes_full['group'] = df_marker_genes_full['group'].astype(str)

#constroi biblioteca de nomes com os arquivos
biblioteca = []

#aba 5A - macrofagos
df_5a = pd.read_excel('fig5_markers_enrichment.xlsx', sheet_name='5A')
for _, row in df_5a.iterrows():
    genes = set(str(row['Genes']).upper().split(';'))
    biblioteca.append({'Nome': row['Cluster'], 'Genes_Ref': genes})

#aba 5EF - goblet, fibrolastos etc - arquivo que liga nomes de células a genes
df_5ef = pd.read_excel('fig5_markers_enrichment.xlsx', sheet_name='5EF')
for col_nome, col_gene in [('source', 'ligand.complex'), ('target', 'receptor.complex')]:
    #agrupa genes por nome de células como no arquivo
    for nome, group in df_5ef.groupby(col_nome):
        genes = set(group[col_gene].astype(str).str.upper().unique())
        biblioteca.append({'Nome': nome, 'Genes_Ref': genes})

#cruzamento
mapeamento_final = {}

for meu_id in df_marker_genes_full['group'].unique():
    #genes dos clusters encontrados
    meus_genes = set(df_marker_genes_full[df_marker_genes_full['group'] == meu_id].head(50)['names'].str.upper())

    melhor_nome = f"Nature_ID_{meu_id}" #default se não achar nada
    maior_intersecao = 0

    for entrada in biblioteca:
        intersecao = len(meus_genes.intersection(entrada['Genes_Ref']))
        if intersecao > maior_intersecao:
            maior_intersecao = intersecao
            melhor_nome = entrada['Nome']

    mapeamento_final[meu_id] = melhor_nome

#aplicação e exibiçao
print("Mapeamento extraído dos arquivos 5A e 5EF:")
for k in sorted(mapeamento_final.keys(), key=int):
    print(f"'{k}' : '{v}'," if (v := mapeamento_final[k]) else "")

#aplica ao objeto
if 'concatenated_sdata' in locals():
    concatenated_sdata["segmentation_counts"].obs["grouped_clusters"] = (
        concatenated_sdata["segmentation_counts"].obs['clusters'].astype(str).map(mapeamento_final).astype('category')
    )

In [ ]:
path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'

# Carrega o objeto na memoria
if 'concatenated_sdata' not in locals():
    import spatialdata as spd
    concatenated_sdata = spd.read_zarr(os.path.join(path_crc, 'sdata_100k_raw.zarr'))

# Recupera os clusters do objeto processado (.h5ad) e transfere para o SpatialData
if 'adata' not in locals():
    import scanpy as sc
    adata = sc.read_h5ad(os.path.join(path_crc, 'adata_full_harmony_umap.h5ad'))

concatenated_sdata["segmentation_counts"].obs['clusters'] = adata.obs['clusters']

mapeamento_final = {
    '0' : 'Goblet', '1' : 'Mast', '2' : 'Goblet', '3' : 'Plasma', '4' : 'Nature_ID_4',
    '5' : 'Enterocyte', '6' : 'Plasma', '7' : 'Plasma', '8' : 'CAF', '9' : 'Nature_ID_9',
    '10' : 'Enterocyte', '11' : 'Neutrophil', '12' : 'Nature_ID_12', '13' : 'Enterocyte',
    '14' : 'Enterocyte', '15' : 'NK', '16' : 'Enterocyte', '17' : 'Goblet',
    '18' : 'Macrophage-SELENOP+', '19' : 'CD4 T cell', '20' : 'CD8 T cell', '21' : 'NK',
    '22' : 'Mature B', '23' : 'CAF', '24' : 'CAF', '25' : 'Enterocyte', '26' : 'CD4 T cell',
    '27' : 'Enterocyte', '28' : 'Endothelial', '29' : 'Enterocyte', '30' : 'Tumor III',
    '31' : 'Enterocyte', '32' : 'Macrophage-SELENOP+', '33' : 'Macrophage-SELENOP+',
    '34' : 'Enterocyte', '35' : 'Macrophage-SELENOP+', '36' : 'Macrophage-SPP1+',
    '37' : 'Macrophage-SELENOP+', '38' : 'Macrophage-SELENOP+', '39' : 'Enterocyte',
    '40' : 'vSM', '41' : 'CAF', '42' : 'CAF', '43' : 'CAF', '44' : 'NK',
    '45' : 'Macrophage-SELENOP+', '46' : 'CAF', '47' : 'CAF', '48' : 'CAF',
    '49' : 'Enterocyte', '50' : 'Macrophage-SPP1+', '51' : 'CAF', '52' : 'NK',
    '53' : 'Enterocyte', '54' : 'CAF', '55' : 'CAF', '56' : 'CAF', '57' : 'CAF',
    '58' : 'CAF', '59' : 'Macrophage-SELENOP+', '60' : 'CAF', '61' : 'Macrophage-SPP1+',
    '62' : 'CAF', '63' : 'Enterocyte', '64' : 'Macrophage-SPP1+', '65' : 'Enterocyte',
    '66' : 'CAF', '67' : 'CAF', '68' : 'CAF', '69' : 'CAF', '70' : 'CAF',
    '71' : 'Enterocyte', '72' : 'Macrophage-SPP1+'
}

# Aplica no objeto spatialdata
concatenated_sdata["segmentation_counts"].obs["grouped_clusters"] = (
    concatenated_sdata["segmentation_counts"].obs['clusters']
    .astype(str)
    .map(mapeamento_final)
    .astype('category')
)

# Exporta o objeto anotado
concatenated_sdata.write(os.path.join(path_crc, 'sdata_full_annotated.zarr'), overwrite=True)

In [ ]:
path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'

#puxa o objeto ja anotado
if 'concatenated_sdata' not in locals():
    import spatialdata as spd
    concatenated_sdata = spd.read_zarr(os.path.join(path_crc, 'sdata_full_annotated.zarr'))

image_elements = list(concatenated_sdata.images.keys())
shape_elements = list(concatenated_sdata.shapes.keys())
extents = []
for i in range(len(image_elements)):
    extent = spd.get_extent(concatenated_sdata, elements=[shape_elements[i]], coordinate_system='downscale_to_hires')
    extents.append(extent)

#gera mapas espaciais
for i in range(len(image_elements)):
    print(f"Plotting: {image_elements[i]}")
    fig, ax = plt.subplots(1, 2, figsize=(18, 8))
    title = image_elements[i].replace("_hires_tissue_image","")

    #lado esquerdo: histologia pura (H&E)
    crop0(concatenated_sdata, crs="downscale_to_hires", bbox=extents[i]).pl.render_images(image_elements[i]).pl.show(ax=ax[0], title=f"{title} - Histologia (H&E)")

    #lado direito: identificação de clusters
    crop0(concatenated_sdata, crs="downscale_to_hires", bbox=extents[i]).pl.render_images(image_elements[i], alpha=0.3).pl.render_shapes(shape_elements[i], color="grouped_clusters").pl.show(ax=ax[1], title=f"{title} - Identificação de Clusters (Full)")

    plt.tight_layout()
    plt.show()

####4.5.4.1. Fig 5

Identification and localization of two macrophage subpopulations in the TME. d, Spatial plots showing the localization of the identified goblet cell subpopulations.

Objetivo: plotar apenas Goblet Cells para comparar visualmente com a literatura

In [ ]:
#lista amostras de interesse - não tem P3 na literatura
samples_to_plot = ['P1', 'P2', 'P5']

#loop de plotagem filtrado
for i in range(len(image_elements)):
    if not any(s in image_elements[i] for s in samples_to_plot):
        continue

    print(f"Plotando Goblet Cells: {image_elements[i]}")
    fig, ax = plt.subplots(1, 2, figsize=(18, 8))
    title = image_elements[i].replace("_hires_tissue_image","")

    #lado esquerdo: Histologia Pura (H&E)
    crop0(concatenated_sdata, crs="downscale_to_hires", bbox=extents[i]).pl.render_images(image_elements[i]).pl.show(ax=ax[0], title=f"{title} - H&E Original")

    #lado direito: Goblet Cells
    crop0(concatenated_sdata, crs="downscale_to_hires", bbox=extents[i])\
        .pl.render_images(image_elements[i], alpha=0.5)\
        .pl.render_shapes(shape_elements[i], color="grouped_clusters", groups=['Goblet'])\
        .pl.show(ax=ax[1], title=f"{title} - Goblet Cells")

    plt.tight_layout()
    plt.show()

Resultados:
* a comparação visual com o resultado obtido com 200k revela que a redução da base de dados resultou em fragmentação severa do sinal biológico, comprometendo a identificação da estrutura. Aqui, é possível observar a continuidade espacial, permitindo a sobreposição à histologia real. Isso valida a necessidade do esforço computacional para a preservação da estrutura biológica;
* a resolução é um fator limitante.

####4.5.4.2. Source Data Fig. 4
Objetivo: validação da proporção de células

é feito o cálculo de composição relativa com value_counts para buscar o total de 200k bins e calcular quanto cada cluster ocupa;
a literatura traz várias amostras, focando em organização espacial e proporções nas maiores linhagens (tumor, stroma, imune) então usa groupby('celltype') para extrair uma média;
foi feito teste qui-quadrado para validar se a distribuição observada segue a distribuição esperada;
como o N é muito grande, qualquer diferença mínima geratia um p-valor menor que 0,05. Por isso, além do p-value, foi feito um resíduo padronizado. Se o resíduo estiver entre -1,96 e +1,96 a proporção é estatisticamente igual à literatura (95% de confiança) - teste de associação e análise de resíduos - o valos na tabela Z quemarca o ponto onde restam apenas 2,5% na cauda é 1,96.
Literatura:

Source Data Fig. 4: Cell type proportion per patient and tissue region, statistical source data.

Fig. 4: Cellular composition of the tumor periphery in each CRC section.

Extended Data Fig. 4 Proportion of cell types identified in three CRC samples. Barplot with the proportion of each of the identified cell types after deconvolution on the three different CRC samples. Colors represent the sample.

In [ ]:
#busca sdata anotado
path_crc = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/objects/crc'
concatenated_sdata = spd.read_zarr(os.path.join(path_crc, 'sdata_full_annotated.zarr'))

#proporções com os nomes anotados da base full
prop_anotada = concatenated_sdata["segmentation_counts"].obs['grouped_clusters'].value_counts(normalize=True) * 100
df_prop_final = prop_anotada.reset_index()
df_prop_final.columns = ['Tipo Celular', 'Proporção pesquisa (%)']

print("Composição do tecido (Base Full):")
display(df_prop_final)

In [ ]:
from scipy.stats import chisquare
import numpy as np

#carrega literatura
df_prop_nat = pd.read_csv('fig4_proportions.csv')
df_prop_nat.columns = [str(c).strip().lower() for c in df_prop_nat.columns]

depara_nature = {
    'CAF': 'Estroma/Fibroblastos', 'Myofibroblast': 'Estroma/Fibroblastos',
    'Fibroblast': 'Estroma/Fibroblastos', 'Proliferating Fibroblast': 'Estroma/Fibroblastos',
    'Vascular Fibroblast': 'Estroma/Fibroblastos',
    'Tumor I': 'Epitelial/Tumoral', 'Tumor II': 'Epitelial/Tumoral', 'Tumor III': 'Epitelial/Tumoral',
    'Tumor IV': 'Epitelial/Tumoral', 'Tumor V': 'Epitelial/Tumoral', 'Epithelial': 'Epitelial/Tumoral',
    'Enterocyte': 'Epitelial/Tumoral', 'Tuft': 'Epitelial/Tumoral',
    'Goblet': 'Células Caliciformes',
    'Macrophage': 'Outras Imunes', 'Proliferating Macrophages': 'Outras Imunes',
    'Plasma': 'Células Plasmáticas', 'Mature B': 'Células Plasmáticas',
    'Smooth Muscle': 'Músculo Liso', 'vSM': 'Músculo Liso', 'Pericytes': 'Músculo Liso'
}

df_prop_nat['categoria_pt'] = df_prop_nat['celltype'].map(depara_nature).fillna('Outras Imunes')
resumo_nat = df_prop_nat.groupby('categoria_pt')['proportion'].sum().reset_index()

total_nat = resumo_nat['proportion'].sum()
resumo_nat['proportion'] = (resumo_nat['proportion'] / total_nat) * 100

#nomes exatos gerados pelo mapeamento da base
depara_pesquisa = {
    'Goblet': 'Células Caliciformes',
    'Tumor III': 'Epitelial/Tumoral',
    'Enterocyte': 'Epitelial/Tumoral',
    'Nature_ID_4': 'Epitelial/Tumoral',
    'Nature_ID_12': 'Epitelial/Tumoral',
    'Nature_ID_9': 'Epitelial/Tumoral',
    'CAF': 'Estroma/Fibroblastos',
    'Plasma': 'Células Plasmáticas',
    'Mature B': 'Células Plasmáticas',
    'vSM': 'Músculo Liso',
    'Macrophage-SELENOP+': 'Outras Imunes',
    'Macrophage-SPP1+': 'Outras Imunes',
    'NK': 'Outras Imunes',
    'Mast': 'Outras Imunes',
    'Neutrophil': 'Outras Imunes',
    'CD4 T cell': 'Outras Imunes',
    'CD8 T cell': 'Outras Imunes',
    'Endothelial': 'Outras Imunes'
}

#contagens brutas
counts_minha = concatenated_sdata["segmentation_counts"].obs['grouped_clusters'].value_counts()
df_minha = counts_minha.reset_index()
df_minha.columns = ['original', 'contagem']
df_minha['categoria_pt'] = df_minha['original'].map(depara_pesquisa).fillna('Outras Imunes')
resumo_minha = df_minha.groupby('categoria_pt')['contagem'].sum().reset_index()

#cruzamento e teste estatístico
tabela_val = pd.merge(resumo_minha, resumo_nat, on='categoria_pt', how='inner')
N_total = tabela_val['contagem'].sum()

tabela_val['esperado_count'] = (tabela_val['proportion'] / 100) * N_total
tabela_val['Pesquisa (%)'] = (tabela_val['contagem'] / N_total) * 100
tabela_val.rename(columns={'proportion': 'Literatura (%)'}, inplace=True)

# Cálculo do Resíduo Padronizado
tabela_val['Resíduo padronizado'] = (tabela_val['contagem'] - tabela_val['esperado_count']) / np.sqrt(tabela_val['esperado_count'])

tabela_val['Validação'] = tabela_val['Resíduo padronizado'].apply(
    lambda x: 'Validado' if abs(x) < 1.96 else 'Diferença significativa'
)

print("TABELA DE VALIDAÇÃO DE PROPORÇÕES (QUI-QUADRADO - BASE FULL)")
tabela_val_exp = tabela_val[['categoria_pt', 'Pesquisa (%)', 'Literatura (%)', 'Resíduo padronizado', 'Validação']]
display(tabela_val_exp)

#export
export_folder = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/tables/crc'
export_file_path = os.path.join(export_folder, 'Source Data Fig. 4 Full.csv')
tabela_val_exp.to_csv(export_file_path, index=False)

In [ ]:
# Preparando dados para o gráfico de comparação

path_tables = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/tables/crc'
tabela_val_exp_8um = pd.read_csv(os.path.join(path_tables, 'Source Data Fig. 4 Full.csv'))

df_plot = tabela_val_exp_8um.melt(id_vars='categoria_pt', value_vars=['Pesquisa (%)', 'Literatura (%)'],
                                  var_name='Fonte', value_name='Porcentagem')

plt.figure(figsize=(12, 6))
sns.barplot(data=df_plot, x='categoria_pt', y='Porcentagem', hue='Fonte', palette=['#1f77b4', '#ff7f0e'])
plt.xticks(rotation=45)
plt.title("Comparação Biológica: Benchmarking 8µm vs Atlas Nature")
plt.ylabel("Proporção do Tecido (%)")
plt.tight_layout()
plt.show()

# Cálculo de Correlação de Spearman (Mede se a ordem de abundância é a mesma)
from scipy.stats import spearmanr
corr, p_val = spearmanr(tabela_val_exp_8um['Pesquisa (%)'], tabela_val_exp_8um['Literatura (%)'])

print(f"Correlação de Ranking (Spearman): {corr:.4f}")

###4.5.6. Análise de expressão gênica diferencial

####4.5.6.2. Wilcoxon

* DESeq2 foi criado para RNA-seq convencional e trata a amostra como uma unidade de observação. É estatisticamente impossível calcular a variância biológica num cenário em que só tem 1 controle e 1 mutado, como será o estudo de caso;
* Diferente do DESeq2, o Wilcoxon trata cada bin/célula como uma observação.No Visium, um único corte pode ter 5k bins de CAF e o Wilcoxon compara a distribuição desses 5k pontos contra os bins de outro corte. Isso dá um poder estatístico grande pelo N alto, permitindo identificar diferenças mesmo com apenas 2 tecidos físicos. Não exige réplicas biológicas por grupo;
* Wilcoxon não assume que os dados seguem uma distribuição normal. Como os dados de TE são frequentemente esparsos, com muitos zeros e alguns valores muito altos, um teste baseado em ranking como o Wilcoxon é mais robusto e menos sensível a outliers que testes paramétricos como o teste t;
* Wilcoxon preserva a variação que existe no tecido, uma vez que não força um pseudobulking, o que é importante para entender se a mutação afeta apenas o que está em contato direto com as células mutadas;
* espera-se que traga mais genes que o DESeq2, porque ignora a variância entre pacientes e foca exclusivamente na variância entre bins

In [ ]:
#prepara os nomes
table.obs['tissue'] = table.obs['sample'].astype(str).str.split('_').str[0].str.strip()

#verificação
print("Grupos encontrados na coluna tissue:")
print(table.obs['tissue'].value_counts())

#filtra CAFs
table_caf = table[table.obs['grouped_clusters'] == 'CAF'].copy()
sc.pp.log1p(table_caf) #normalização logarítmica ajuda a comprimir a escala e reduzir a influência de bins com outliers

#Wilcoxon (cancer vs normal)
sc.tl.rank_genes_groups(
    table_caf,
    groupby='tissue',
    reference='Normal',
    method='wilcoxon',
    key_added='wilcoxon_cancer_vs_normal'
)

df_wilcoxon = sc.get.rank_genes_groups_df(table_caf, group='Cancer', key='wilcoxon_cancer_vs_normal')
df_wilcoxon_sig = df_wilcoxon[(df_wilcoxon['pvals_adj'] < 0.05) & (df_wilcoxon['logfoldchanges'] > 0)]

print(f"\nWilcoxon: {len(df_wilcoxon_sig)} genes significativos.")
display(df_wilcoxon_sig.head(10))

In [ ]:
#export
export_path = '/content/drive/MyDrive/projects/visium-hd-spatial-transcriptomics/tables/crc'
genes_significativos.to_csv(os.path.join(export_path, 'wilcoxon_cancer_vs_normal_full.csv'))

Resultados:

*a convergência dos resultados estatísticos evidencia a robustez do pipeline analítico:
 *preservação de Marcadores Canônicos: os principais biomarcadores identificados pela literatura (SFRP4, ADAM12, COL11A1) foram detectados com elevada significância em ambos os métodos estatísticos aplicados. Tal consistência indica que o sinal biológico é resiliente a variações metodológicas;
 * convergência estatística: A elevada sobreposição de resultados, com uma interseção de 879 genes significativos, demonstra que os métodos convergem na caracterização da resposta inflamatória e no remodelamento do estroma tumoral.
* o pipeline demonstrou alta fidelidade na recuperação das identidades celulares e nichos moleculares descritos em estudos de referência (Oliveira et al., 2025), validando sua aplicação para estudos de caso complexos.